# Publication notes

This is the original exploratory workflow with saved outputs removed and paths made configurable. Collection and analytical logic is retained. Source websites and identifiers may change; notebooks 01–02 are an inspected research workflow, not a guaranteed unattended scraper. Review collection permissions and source-specific settings before recollection. Do not run against your only copy of historical data: original export cells can overwrite files.

Run from a clean kernel in order. Data is local and excluded from GitHub. Historical saved review text and reviewer details are not published as notebook outputs.


In [ ]:
from pathlib import Path
import os

# Configure KBEAUTY_PROJECT_ROOT to keep data outside the cloned repository.
_working_dir = Path.cwd().resolve()
_default_root = _working_dir.parent if _working_dir.name == "notebooks" else _working_dir
PROJECT_ROOT = Path(os.environ.get("KBEAUTY_PROJECT_ROOT", str(_default_root))).expanduser().resolve()
# Original collection cells use relative data/ paths.
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)


# K-Beauty Consumer Insights — Full Data Collection

This notebook collects and structures product and consumer review data
for 12 K-beauty skincare products across four product segments.

## Final relational tables

1. `products`
2. `reviews`
3. `ingredients`
4. `review_skin_concerns`

The collection target is up to 300 of the most recent reviews per product.

In [ ]:
import pandas as pd
import requests
import json
import re
import time
from pathlib import Path
from bs4 import BeautifulSoup

print("Libraries imported successfully!")

## 1. Product Configuration

Define the 12 products included in the analysis and assign a unique product ID to each product.

In [ ]:
products_config = [
    # CLEANSERS
    {
        "product_id": "BOJ_CLEANSER_01",
        "brand": "Beauty of Joseon",
        "product_name": "Green Plum Refreshing Cleanser",
        "segment": "Cleanser"
    },
    {
        "product_id": "COSRX_CLEANSER_01",
        "brand": "COSRX",
        "product_name": "Low pH Good Morning Gel Cleanser",
        "segment": "Cleanser"
    },
    {
        "product_id": "LANEIGE_CLEANSER_01",
        "brand": "LANEIGE",
        "product_name": "Water Bank Gentle Gel Cleanser",
        "segment": "Cleanser"
    },

    # MOISTURIZERS
    {
        "product_id": "BOJ_MOISTURIZER_01",
        "brand": "Beauty of Joseon",
        "product_name": "Dynasty Cream",
        "segment": "Moisturizer"
    },
    {
        "product_id": "COSRX_MOISTURIZER_01",
        "brand": "COSRX",
        "product_name": "Advanced Snail 92 All in One Cream",
        "segment": "Moisturizer"
    },
    {
        "product_id": "LANEIGE_MOISTURIZER_01",
        "brand": "LANEIGE",
        "product_name": "Water Bank Blue Hyaluronic Cream Moisturizer",
        "segment": "Moisturizer"
    },

    # SERUMS
    {
        "product_id": "BOJ_SERUM_01",
        "brand": "Beauty of Joseon",
        "product_name": "Glow Serum: Propolis + Niacinamide",
        "segment": "Serum"
    },
    {
        "product_id": "COSRX_SERUM_01",
        "brand": "COSRX",
        "product_name": "The 6 Peptide Skin Booster Serum",
        "segment": "Serum"
    },
    {
        "product_id": "LANEIGE_SERUM_01",
        "brand": "LANEIGE",
        "product_name": "Water Bank Blue Hyaluronic Serum",
        "segment": "Serum"
    },

    # TONERS
    {
        "product_id": "BOJ_TONER_01",
        "brand": "Beauty of Joseon",
        "product_name": "Ginseng Essence Water",
        "segment": "Toner / Essence"
    },
    {
        "product_id": "COSRX_TONER_01",
        "brand": "COSRX",
        "product_name": "Full Fit Propolis Synergy Toner",
        "segment": "Toner"
    },
    {
        "product_id": "LANEIGE_TONER_01",
        "brand": "LANEIGE",
        "product_name": "Cream Skin Milky Toner",
        "segment": "Toner"
    }
]

products_config_df = pd.DataFrame(products_config)

print("Number of products:", len(products_config_df))
products_config_df

In [ ]:
# Add the URLs extracted from K-Beauty Project.xltx.
# Match by permanent product ID, rather than list position.

product_urls = {
    # CLEANSERS
    "BOJ_CLEANSER_01":
        "https://beautyofjoseon.com/products/green-plum-refreshing-cleanser",
    "COSRX_CLEANSER_01":
        "https://www.cosrx.com/collections/cleansers/products/low-ph-good-morning-gel-cleanser?variant=37517677953180",
    "LANEIGE_CLEANSER_01":
        "https://us.laneige.com/products/water-bank-gentle-gel-cleanser?variant=45866099671092",

    # MOISTURIZERS
    "BOJ_MOISTURIZER_01":
        "https://beautyofjoseon.com/products/dynasty-cream",
    "COSRX_MOISTURIZER_01":
        "https://www.cosrx.com/collections/moisturizer-cream/products/advanced-snail-92-all-in-one-cream",
    "LANEIGE_MOISTURIZER_01":
        "https://us.laneige.com/products/water-bank-blue-hyaluronic-cream-moisturizer?variant=48218677739572",

    # SERUMS
    "BOJ_SERUM_01":
        "https://beautyofjoseon.com/products/glow-serum-propolis-niacinamide",
    "COSRX_SERUM_01":
        "https://www.cosrx.com/collections/serums-essences/products/the-6-peptide-skin-booster-serum-bundle",
    "LANEIGE_SERUM_01":
        "https://us.laneige.com/products/water-bank-blue-hyaluronic-serum?variant=39875620864052",

    # TONERS / ESSENCES
    "BOJ_TONER_01":
        "https://beautyofjoseon.com/products/ginseng-essence-water",
    "COSRX_TONER_01":
        "https://cosrx.com/collections/toners/products/full-fit-propolis-synergy-toner",
    "LANEIGE_TONER_01":
        "https://us.laneige.com/products/cream-skin-toner-moisturizer?variant=48218688585780",
}

# Check the configuration before changing it.
config_ids = [product["product_id"] for product in products_config]

assert len(config_ids) == 12, "Expected 12 products."
assert len(set(config_ids)) == 12, "Duplicate product IDs found."
assert set(config_ids) == set(product_urls), (
    f"Products missing URLs: {set(config_ids) - set(product_urls)}; "
    f"URLs with unknown IDs: {set(product_urls) - set(config_ids)}"
)

# Add URLs and refresh the display table.
for product in products_config:
    product["product_url"] = product_urls[product["product_id"]]

products_config_df = pd.DataFrame(products_config)

print("Products configured:", len(products_config_df))
print("Products with URLs:", products_config_df["product_url"].notna().sum())

products_config_df[
    ["product_id", "brand", "product_name", "segment", "product_url"]
]

In [ ]:
# Select the remaining Beauty of Joseon products.
# Ginseng Essence Water was already collected in the test notebook.

boj_products_to_collect = [
    product
    for product in products_config
    if product["brand"] == "Beauty of Joseon"
    and product["product_id"] != "BOJ_TONER_01"
]

print("Beauty of Joseon products to collect:", len(boj_products_to_collect))

pd.DataFrame(boj_products_to_collect)[
    ["product_id", "product_name", "product_url"]
]

In [ ]:
def find_product_json_ld(soup):
    """Find Product data without relying on a fixed script position."""

    def find_products(value):
        if isinstance(value, list):
            for item in value:
                yield from find_products(item)

        elif isinstance(value, dict):
            types = value.get("@type", [])
            if isinstance(types, str):
                types = [types]

            if "Product" in types:
                yield value

            if "@graph" in value:
                yield from find_products(value["@graph"])

    products = []

    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.get_text())
        except json.JSONDecodeError:
            continue

        products.extend(find_products(data))

    if len(products) != 1:
        raise ValueError(
            f"Expected one Product block; found {len(products)}. "
            "Inspect the page before continuing."
        )

    return products[0]


# Test on the cleanser.
test_product = next(
    product for product in products_config
    if product["product_id"] == "BOJ_CLEANSER_01"
)

response = requests.get(test_product["product_url"], timeout=30)
response.raise_for_status()

test_soup = BeautifulSoup(response.text, "html.parser")
test_product_data = find_product_json_ld(test_soup)

print(json.dumps(test_product_data, indent=2, ensure_ascii=False))

In [ ]:
def build_product_record(product_config, product_data):
    """Build a product record using the offer matching its main SKU."""
    sku = product_data.get("sku")
    offers = product_data.get("offers", [])

    if isinstance(offers, dict):
        offers = [offers]

    matching_offers = [
        offer for offer in offers
        if sku and offer.get("sku") == sku
    ]

    if len(matching_offers) != 1:
        raise ValueError(
            f"Expected one offer matching SKU {sku}; "
            f"found {len(matching_offers)}."
        )

    offer = matching_offers[0]

    return {
        "product_id": product_config["product_id"],
        "brand": product_config["brand"],
        "product_name": product_data["name"],
        "segment": product_config["segment"],
        "sku": sku,
        "current_price": float(offer["price"]),
        "currency": offer["priceCurrency"],
        "description": (product_data.get("description") or "").strip(),
        "product_url": product_config["product_url"],
        "variant_url": offer.get("url")
    }


test_product_record = build_product_record(
    test_product,
    test_product_data
)

pd.DataFrame([test_product_record])

In [ ]:
def extract_custom_metafields(soup):
    """Read the product's customMetafields from its page scripts."""
    for script in soup.find_all("script"):
        script_text = script.string or script.get_text()
        match = re.search(r"\bcustomMetafields\s*:", script_text)

        if match:
            remaining_text = script_text[match.end():].lstrip()
            metafields, _ = json.JSONDecoder().raw_decode(remaining_text)

            if not isinstance(metafields, dict):
                raise ValueError("Expected customMetafields to be a dictionary.")

            return metafields

    raise ValueError("customMetafields was not found on this product page.")


test_metafields = extract_custom_metafields(test_soup)

# Inspect the actual values before adding them to our product record.
for field in [
    "key_ingredients",
    "key_benefits",
    "skin_concern",
    "all_ingredients"
]:
    print(f"\n{field}:")
    print(json.dumps(test_metafields.get(field), indent=2, ensure_ascii=False))

In [ ]:
# Add the extracted product attributes.
test_product_record.update({
    "key_ingredients": test_metafields.get("key_ingredients"),
    "key_benefits": test_metafields.get("key_benefits"),
    "skin_concern": test_metafields.get("skin_concern")
})

# Keep the original ingredient text for reference.
ingredient_versions = test_metafields.get("all_ingredients")

if (
    not isinstance(ingredient_versions, list)
    or not ingredient_versions
    or not isinstance(ingredient_versions[0], str)
    or not ingredient_versions[0].strip()
):
    raise ValueError("Expected a non-empty ingredient list.")

test_ingredients_raw = ingredient_versions[0]

# Suitable for the cleanser's observed format.
ingredient_names = [
    ingredient.strip()
    for ingredient in re.split(r",\s+", test_ingredients_raw.strip())
    if ingredient.strip()
]

test_ingredients_df = pd.DataFrame({
    "product_id": test_product_record["product_id"],
    "ingredient_position": range(1, len(ingredient_names) + 1),
    "ingredient_name": ingredient_names
})

print("Number of ingredients:", len(test_ingredients_df))
print(
    "1,2-Hexanediol preserved:",
    "1,2-Hexanediol" in ingredient_names
)

test_ingredients_df

In [ ]:
# Find the product IDs attached to Okendo review widgets.
okendo_product_ids = sorted({
    element["data-oke-reviews-product-id"]
    for element in test_soup.find_all(
        attrs={"data-oke-reviews-product-id": True}
    )
})

print("Okendo product IDs found:", okendo_product_ids)

# If the attribute isn't present, show relevant script/HTML snippets.
if not okendo_product_ids:
    page_html = str(test_soup)

    for match in list(
        re.finditer(r"shopify-\d+", page_html)
    )[:5]:
        start = max(0, match.start() - 150)
        end = match.end() + 150

        print("\n--- Possible product ID ---")
        print(page_html[start:end])

In [ ]:
cleanser_reviews_url = (
    "https://api.okendo.io/v1/stores/"
    "121d630c-0612-42f9-8f88-70cb5ef6a19b/"
    "products/shopify-7312924803237/reviews"
)

reviews_response = requests.get(
    cleanser_reviews_url,
    params={
        "limit": 5,
        "locale": "en",
        "orderBy": "date desc"
    },
    timeout=30
)
reviews_response.raise_for_status()

cleanser_reviews_data = reviews_response.json()
preview_reviews = cleanser_reviews_data.get("reviews", [])

print("Reviews returned:", len(preview_reviews))

pd.DataFrame([
    {
        "review_date": review.get("dateCreated"),
        "rating": review.get("rating"),
        "review_title": review.get("title")
    }
    for review in preview_reviews
])

In [ ]:
import requests
import time

def collect_reviews(reviews_url, product_id, max_reviews=300):
    current_url = reviews_url
    params = {
        "limit": 25,
        "locale": "en",
        "orderBy": "date desc"
    }

    collected = []
    seen_ids = set()
    visited_urls = set()

    while current_url and len(collected) < max_reviews:
        if current_url in visited_urls:
            raise ValueError("Repeated pagination URL detected.")

        visited_urls.add(current_url)

        response = requests.get(
            current_url,
            params=params,
            timeout=30
        )
        response.raise_for_status()
        data = response.json()

        reviews = data.get("reviews", [])
        if not reviews:
            break

        for review in reviews:
            review_id = review.get("reviewId")

            if not review_id:
                raise ValueError("A review is missing its reviewId.")

            if review_id not in seen_ids:
                seen_ids.add(review_id)
                collected.append({
                    **review,
                    "relational_product_id": product_id
                })

            if len(collected) >= max_reviews:
                break

        print(f"Collected {len(collected)} / {max_reviews} reviews")

        # Preserve the API's /v1 prefix for relative pagination links.
        next_url = data.get("nextUrl")

        if not next_url:
            current_url = None
        elif next_url.startswith(("https://", "http://")):
            current_url = next_url
        elif next_url.startswith("/v1/"):
            current_url = "https://api.okendo.io" + next_url
        else:
            current_url = (
                "https://api.okendo.io/v1/"
                + next_url.lstrip("/")
            )

        # Pagination links already include the query parameters.
        params = None

        if current_url and len(collected) < max_reviews:
            time.sleep(0.5)

    return collected


cleanser_reviews_url = (
    "https://api.okendo.io/v1/stores/"
    "121d630c-0612-42f9-8f88-70cb5ef6a19b/"
    "products/shopify-7312924803237/reviews"
)

cleanser_reviews = collect_reviews(
    reviews_url=cleanser_reviews_url,
    product_id="BOJ_CLEANSER_01",
    max_reviews=300
)

print("\nFinal number of reviews:", len(cleanser_reviews))

In [ ]:
def clean_review(review):
    reviewer = review.get("reviewer") or {}
    attributes = reviewer.get("attributes") or []

    reviewer_attributes = {
        attribute["title"]: attribute.get("value")
        for attribute in attributes
        if attribute.get("title")
    }

    return {
        "review_id": review.get("reviewId"),
        "product_id": review.get("relational_product_id"),
        "review_date": review.get("dateCreated"),
        "rating": review.get("rating"),
        "review_title": review.get("title"),
        "review_text": review.get("body"),
        "reviewer_name": reviewer.get("displayName"),
        "age": reviewer_attributes.get("How old are you?"),
        "skin_type": reviewer_attributes.get("What is your skin type?"),
        "skin_concerns": reviewer_attributes.get(
            "What are your skin concerns?"
        ),
        "verified": reviewer.get("isVerified"),
        "recommended": review.get("isRecommended"),
        "incentivized": review.get("isIncentivized"),
        "helpful_count": review.get("helpfulCount"),
        "unhelpful_count": review.get("unhelpfulCount"),
        "language": review.get("languageCode")
    }


cleanser_reviews_df = pd.DataFrame([
    clean_review(review)
    for review in cleanser_reviews
])

cleanser_reviews_df["review_date"] = pd.to_datetime(
    cleanser_reviews_df["review_date"],
    errors="coerce",
    utc=True
)

print("Reviews:", len(cleanser_reviews_df))
print(
    "Duplicate review IDs:",
    cleanser_reviews_df["review_id"].duplicated().sum()
)
print(
    "Missing or invalid dates:",
    cleanser_reviews_df["review_date"].isna().sum()
)
print(
    "Newest-first order:",
    cleanser_reviews_df["review_date"].is_monotonic_decreasing
)
print("Newest review:", cleanser_reviews_df["review_date"].max())
print("Oldest review:", cleanser_reviews_df["review_date"].min())

cleanser_reviews_df.head()

In [ ]:
# Separate reviewers' skin concerns into their own table.
cleanser_review_skin_concerns_df = (
    cleanser_reviews_df[["review_id", "skin_concerns"]]
    .explode("skin_concerns")
    .dropna(subset=["skin_concerns"])
    .rename(columns={"skin_concerns": "skin_concern"})
    .copy()
)

# Remove blank values and duplicate pairs.
cleanser_review_skin_concerns_df["skin_concern"] = (
    cleanser_review_skin_concerns_df["skin_concern"]
    .astype(str)
    .str.strip()
)

cleanser_review_skin_concerns_df = (
    cleanser_review_skin_concerns_df[
        cleanser_review_skin_concerns_df["skin_concern"] != ""
    ]
    .drop_duplicates(subset=["review_id", "skin_concern"])
    .reset_index(drop=True)
)

# Keep all 300 reviews, including those without skin concerns.
cleanser_reviews_clean_df = cleanser_reviews_df.drop(
    columns=["skin_concerns"]
).copy()

print("Reviews retained:", len(cleanser_reviews_clean_df))
print("Review–concern pairs:", len(cleanser_review_skin_concerns_df))
print(
    "Reviews with skin concerns:",
    cleanser_review_skin_concerns_df["review_id"].nunique()
)
print(
    "Orphan review concerns:",
    (~cleanser_review_skin_concerns_df["review_id"].isin(
        cleanser_reviews_clean_df["review_id"]
    )).sum()
)

cleanser_review_skin_concerns_df.head(10)

In [ ]:
# Assemble the product and ingredient tables.
cleanser_products_df = pd.DataFrame([test_product_record])
cleanser_ingredients_df = test_ingredients_df.copy()

# Check the relationships between tables.
orphan_reviews = (
    ~cleanser_reviews_clean_df["product_id"].isin(
        cleanser_products_df["product_id"]
    )
).sum()

orphan_ingredients = (
    ~cleanser_ingredients_df["product_id"].isin(
        cleanser_products_df["product_id"]
    )
).sum()

orphan_concerns = (
    ~cleanser_review_skin_concerns_df["review_id"].isin(
        cleanser_reviews_clean_df["review_id"]
    )
).sum()

print("Products:", len(cleanser_products_df))
print("Reviews:", len(cleanser_reviews_clean_df))
print("Ingredients:", len(cleanser_ingredients_df))
print("Review–concern pairs:", len(cleanser_review_skin_concerns_df))

print("\nOrphan reviews:", orphan_reviews)
print("Orphan ingredients:", orphan_ingredients)
print("Orphan review concerns:", orphan_concerns)

# Confirm these tables belong to the cleanser.
assert cleanser_products_df["product_id"].tolist() == [
    "BOJ_CLEANSER_01"
]
assert len(cleanser_reviews_clean_df) == 300
assert cleanser_reviews_clean_df["review_id"].is_unique
assert orphan_reviews == 0
assert orphan_ingredients == 0
assert orphan_concerns == 0

print("\nAll checks passed.")

In [ ]:
from pathlib import Path

cleanser_output_folder = Path(
    "data/processed/BOJ_CLEANSER_01"
)
cleanser_output_folder.mkdir(parents=True, exist_ok=True)

tables_to_save = {
    "products.csv": cleanser_products_df,
    "reviews.csv": cleanser_reviews_clean_df,
    "ingredients.csv": cleanser_ingredients_df,
    "review_skin_concerns.csv": cleanser_review_skin_concerns_df
}

for filename, table in tables_to_save.items():
    file_path = cleanser_output_folder / filename
    table.to_csv(file_path, index=False)

    # Confirm the saved file contains the expected number of rows.
    saved_table = pd.read_csv(file_path)
    assert len(saved_table) == len(table), (
        f"Row count mismatch in {filename}"
    )

    print(f"{filename}: {len(saved_table)} rows saved")

print("\nSaved to:", cleanser_output_folder.resolve())

In [ ]:
# Select Dynasty Cream from the configuration.
dynasty_config = next(
    product for product in products_config
    if product["product_id"] == "BOJ_MOISTURIZER_01"
)

# Download its product page.
dynasty_response = requests.get(
    dynasty_config["product_url"],
    timeout=30
)
dynasty_response.raise_for_status()

dynasty_soup = BeautifulSoup(
    dynasty_response.text,
    "html.parser"
)

# Extract product details and custom fields.
dynasty_product_data = find_product_json_ld(dynasty_soup)
dynasty_metafields = extract_custom_metafields(dynasty_soup)

print("Product:", dynasty_product_data.get("name"))
print("Main SKU:", dynasty_product_data.get("sku"))

print("\nOffers:")
print(json.dumps(
    dynasty_product_data.get("offers"),
    indent=2,
    ensure_ascii=False
))

print("\nIngredients:")
print(json.dumps(
    dynasty_metafields.get("all_ingredients"),
    indent=2,
    ensure_ascii=False
))

In [ ]:
# Build the product record using the offer matching the main SKU.
dynasty_product_record = build_product_record(
    dynasty_config,
    dynasty_product_data
)

dynasty_product_record.update({
    "key_ingredients": dynasty_metafields.get("key_ingredients"),
    "key_benefits": dynasty_metafields.get("key_benefits"),
    "skin_concern": dynasty_metafields.get("skin_concern")
})

dynasty_products_df = pd.DataFrame([dynasty_product_record])

# Use the first ingredient version, preserving names like 1,2-Hexanediol.
dynasty_ingredients_raw = dynasty_metafields["all_ingredients"][0]

dynasty_ingredient_names = [
    ingredient.strip()
    for ingredient in re.split(r",\s+", dynasty_ingredients_raw.strip())
    if ingredient.strip()
]

dynasty_ingredients_df = pd.DataFrame({
    "product_id": dynasty_config["product_id"],
    "ingredient_position": range(1, len(dynasty_ingredient_names) + 1),
    "ingredient_name": dynasty_ingredient_names
})

print("Product:", dynasty_product_record["product_name"])
print("SKU:", dynasty_product_record["sku"])
print(
    "Price:",
    dynasty_product_record["current_price"],
    dynasty_product_record["currency"]
)
print("Ingredients:", len(dynasty_ingredients_df))

dynasty_ingredients_df

In [ ]:
def collect_and_save_product(product_record, ingredients_df, reviews_url):
    product_id = product_record["product_id"]

    # 1. Collect up to 300 reviews, newest first.
    raw_reviews = collect_reviews(
        reviews_url=reviews_url,
        product_id=product_id,
        max_reviews=300
    )

    if not raw_reviews:
        raise ValueError("No reviews returned. Check the review URL.")

    # 2. Clean reviews using the existing function.
    reviews_df = pd.DataFrame([
        clean_review(review) for review in raw_reviews
    ])

    reviews_df["review_date"] = pd.to_datetime(
        reviews_df["review_date"],
        errors="coerce",
        utc=True
    )

    # 3. Create the review–skin concern table.
    concerns_df = (
        reviews_df[["review_id", "skin_concerns"]]
        .explode("skin_concerns")
        .dropna(subset=["skin_concerns"])
        .rename(columns={"skin_concerns": "skin_concern"})
        .copy()
    )

    concerns_df["skin_concern"] = (
        concerns_df["skin_concern"].astype(str).str.strip()
    )

    concerns_df = (
        concerns_df[concerns_df["skin_concern"] != ""]
        .drop_duplicates(["review_id", "skin_concern"])
        .reset_index(drop=True)
    )

    reviews_df = reviews_df.drop(columns=["skin_concerns"])
    products_df = pd.DataFrame([product_record])
    ingredients_df = ingredients_df.copy()

    # 4. Validate before saving.
    assert reviews_df["review_id"].notna().all(), "Missing review IDs."
    assert reviews_df["review_id"].is_unique, "Duplicate review IDs."
    assert reviews_df["review_date"].notna().all(), "Invalid review dates."
    assert reviews_df["review_date"].is_monotonic_decreasing, (
        "Reviews are not ordered newest first."
    )
    assert reviews_df["product_id"].eq(product_id).all(), (
        "Reviews contain an incorrect product ID."
    )
    assert not ingredients_df.empty, "No ingredients found."
    assert ingredients_df["product_id"].eq(product_id).all(), (
        "Ingredients contain an incorrect product ID."
    )
    assert concerns_df["review_id"].isin(reviews_df["review_id"]).all(), (
        "Orphan review concerns found."
    )

    # 5. Save each table in this product's folder.
    output_folder = Path("data/processed") / product_id
    output_folder.mkdir(parents=True, exist_ok=True)

    tables = {
        "products": products_df,
        "reviews": reviews_df,
        "ingredients": ingredients_df,
        "review_skin_concerns": concerns_df
    }

    for name, table in tables.items():
        file_path = output_folder / f"{name}.csv"
        table.to_csv(file_path, index=False)

        saved = pd.read_csv(file_path)
        assert len(saved) == len(table), f"Row count mismatch: {name}"

        print(f"{name}.csv: {len(saved)} rows saved")

    print("\nAll checks passed.")
    print("Saved to:", output_folder.resolve())

    return tables


# Start from the first page by omitting the copied pagination cursor.
dynasty_reviews_url = (
    "https://api.okendo.io/v1/stores/"
    "121d630c-0612-42f9-8f88-70cb5ef6a19b/"
    "products/shopify-6874076479653/reviews"
)

dynasty_tables = collect_and_save_product(
    product_record=dynasty_product_record,
    ingredients_df=dynasty_ingredients_df,
    reviews_url=dynasty_reviews_url
)

In [ ]:
glow_config = next(
    product for product in products_config
    if product["product_id"] == "BOJ_SERUM_01"
)

glow_response = requests.get(
    glow_config["product_url"],
    timeout=30
)
glow_response.raise_for_status()

glow_soup = BeautifulSoup(glow_response.text, "html.parser")
glow_product_data = find_product_json_ld(glow_soup)
glow_metafields = extract_custom_metafields(glow_soup)

glow_product_record = build_product_record(
    glow_config,
    glow_product_data
)

glow_product_record.update({
    "key_ingredients": glow_metafields.get("key_ingredients"),
    "key_benefits": glow_metafields.get("key_benefits"),
    "skin_concern": glow_metafields.get("skin_concern")
})

ingredient_versions = glow_metafields.get("all_ingredients")

if (
    not isinstance(ingredient_versions, list)
    or not ingredient_versions
    or not isinstance(ingredient_versions[0], str)
    or not ingredient_versions[0].strip()
):
    raise ValueError("Expected a non-empty ingredient list.")

glow_ingredients_raw = ingredient_versions[0]

glow_ingredient_names = [
    ingredient.strip()
    for ingredient in re.split(r",\s+", glow_ingredients_raw.strip())
    if ingredient.strip()
]

glow_ingredients_df = pd.DataFrame({
    "product_id": glow_config["product_id"],
    "ingredient_position": range(1, len(glow_ingredient_names) + 1),
    "ingredient_name": glow_ingredient_names
})

print("Product:", glow_product_record["product_name"])
print("SKU:", glow_product_record["sku"])
print(
    "Price:",
    glow_product_record["current_price"],
    glow_product_record["currency"]
)
print("Ingredients:", len(glow_ingredients_df))
print("\nOriginal ingredient text:\n", glow_ingredients_raw)

In [ ]:
# Omit the pagination cursor so collection starts on the first page.
glow_reviews_url = (
    "https://api.okendo.io/v1/stores/"
    "121d630c-0612-42f9-8f88-70cb5ef6a19b/"
    "products/shopify-6805117370533/reviews"
)

glow_tables = collect_and_save_product(
    product_record=glow_product_record,
    ingredients_df=glow_ingredients_df,
    reviews_url=glow_reviews_url
)

In [ ]:
# 1. Select and download Ginseng Essence Water.
ginseng_config = next(
    product for product in products_config
    if product["product_id"] == "BOJ_TONER_01"
)

ginseng_response = requests.get(
    ginseng_config["product_url"],
    timeout=30
)
ginseng_response.raise_for_status()

ginseng_soup = BeautifulSoup(
    ginseng_response.text,
    "html.parser"
)

# 2. Select the detailed Product block using its confirmed SKU.
ginseng_matches = [
    product
    for product in list_product_blocks(ginseng_soup)
    if product.get("sku") == "82BS002"
]

if len(ginseng_matches) != 1:
    raise ValueError(
        "Expected one Ginseng product block with SKU 82BS002; "
        f"found {len(ginseng_matches)}."
    )

ginseng_product_data = ginseng_matches[0]
ginseng_metafields = extract_custom_metafields(ginseng_soup)

# 3. Build the product record.
ginseng_product_record = build_product_record(
    ginseng_config,
    ginseng_product_data
)

ginseng_product_record.update({
    "key_ingredients": ginseng_metafields.get("key_ingredients"),
    "key_benefits": ginseng_metafields.get("key_benefits"),
    "skin_concern": ginseng_metafields.get("skin_concern")
})

# 4. Extract the first ingredient list.
ingredient_versions = ginseng_metafields.get("all_ingredients")

if (
    not isinstance(ingredient_versions, list)
    or not ingredient_versions
    or not isinstance(ingredient_versions[0], str)
    or not ingredient_versions[0].strip()
):
    raise ValueError("Expected a non-empty ingredient list.")

ginseng_ingredients_raw = ingredient_versions[0]

ginseng_ingredient_names = [
    ingredient.strip()
    for ingredient in re.split(
        r",\s+",
        ginseng_ingredients_raw.strip()
    )
    if ingredient.strip()
]

ginseng_ingredients_df = pd.DataFrame({
    "product_id": ginseng_config["product_id"],
    "ingredient_position": range(
        1, len(ginseng_ingredient_names) + 1
    ),
    "ingredient_name": ginseng_ingredient_names
})

print("Product:", ginseng_product_record["product_name"])
print("SKU:", ginseng_product_record["sku"])
print(
    "Price:",
    ginseng_product_record["current_price"],
    ginseng_product_record["currency"]
)
print("Ingredients:", len(ginseng_ingredients_df))
print("\nOriginal ingredient text:", ginseng_ingredients_raw)

# 5. Collect reviews from the first page.
# collect_reviews applies orderBy="date desc".
ginseng_reviews_url = (
    "https://api.okendo.io/v1/stores/"
    "121d630c-0612-42f9-8f88-70cb5ef6a19b/"
    "products/shopify-6805137195173/reviews"
)

# 6. Clean, validate, and save all four tables.
ginseng_tables = collect_and_save_product(
    product_record=ginseng_product_record,
    ingredients_df=ginseng_ingredients_df,
    reviews_url=ginseng_reviews_url
)

In [ ]:
boj_product_ids = [
    "BOJ_CLEANSER_01",
    "BOJ_MOISTURIZER_01",
    "BOJ_SERUM_01",
    "BOJ_TONER_01"
]

table_names = [
    "products",
    "reviews",
    "ingredients",
    "review_skin_concerns"
]

# Read and combine the four products.
boj_tables = {}

for table_name in table_names:
    frames = [
        pd.read_csv(
            Path("data/processed") / product_id / f"{table_name}.csv"
        )
        for product_id in boj_product_ids
    ]
    boj_tables[table_name] = pd.concat(frames, ignore_index=True)

# Validate the combined relationships.
products = boj_tables["products"]
reviews = boj_tables["reviews"]
ingredients = boj_tables["ingredients"]
concerns = boj_tables["review_skin_concerns"]

assert products["product_id"].is_unique, "Duplicate products."
assert set(products["product_id"]) == set(boj_product_ids)
assert reviews["review_id"].notna().all(), "Missing review IDs."
assert reviews["review_id"].is_unique, (
    "Review IDs overlap across products. Inspect before combining."
)
assert reviews["product_id"].isin(products["product_id"]).all()
assert ingredients["product_id"].isin(products["product_id"]).all()
assert concerns["review_id"].isin(reviews["review_id"]).all()
assert not concerns.duplicated(["review_id", "skin_concern"]).any()

# Save the combined brand tables.
boj_output_folder = Path("data/processed/beauty_of_joseon")
boj_output_folder.mkdir(parents=True, exist_ok=True)

for table_name, table in boj_tables.items():
    file_path = boj_output_folder / f"{table_name}.csv"
    table.to_csv(file_path, index=False)
    assert len(pd.read_csv(file_path)) == len(table)
    print(f"{table_name}.csv: {len(table)} rows saved")

print("\nAll combined checks passed.")
print("Saved to:", boj_output_folder.resolve())

In [ ]:
cosrx_cleanser_reviews_url = (
    "https://api-cdn.yotpo.com/v3/storefront/store/"
    "FHrLUCZg4k9lvDprUIkr92ql2rzsUDWvqR67HEcq/"
    "product/6101923299484/reviews"
)

cosrx_response = requests.get(
    cosrx_cleanser_reviews_url,
    params={
        "page": 1,
        "perPage": 9,
        "sort": "date,smart,images,badge,rating"
    },
    timeout=30
)
cosrx_response.raise_for_status()

cosrx_reviews_data = cosrx_response.json()

# Inspect the response structure before building the collector.
print(json.dumps(
    cosrx_reviews_data,
    indent=2,
    ensure_ascii=False
)[:18000])

In [ ]:
def collect_yotpo_reviews(reviews_url, product_id, max_reviews=300):
    collected = []
    seen_ids = set()
    page = 1
    per_page = 9
    previous_date = None

    while len(collected) < max_reviews:
        response = requests.get(
            reviews_url,
            params={
                "page": page,
                "perPage": per_page,
                "sort": "date,smart,images,badge,rating"
            },
            timeout=30
        )
        response.raise_for_status()
        data = response.json()

        pagination = data["pagination"]
        reviews = data["reviews"]

        if int(pagination["page"]) != page:
            raise ValueError("The API returned an unexpected page.")

        if not reviews:
            break

        added = 0

        for review in reviews:
            review_id = review.get("id")
            if review_id is None:
                raise ValueError("A review is missing its ID.")

            if review_id in seen_ids:
                continue

            # Yotpo's observed timestamps have no timezone.
            # Check their order without assigning one.
            review_date = pd.to_datetime(
                review.get("createdAt"),
                errors="coerce"
            )

            if pd.isna(review_date):
                raise ValueError("A review has a missing or invalid date.")

            if previous_date is not None and review_date > previous_date:
                raise ValueError(
                    f"Newest-first ordering failed on page {page}. "
                    "Check the website sorting before continuing."
                )

            previous_date = review_date
            seen_ids.add(review_id)
            collected.append({
                **review,
                "relational_product_id": product_id
            })
            added += 1

            if len(collected) >= max_reviews:
                break

        print(f"Collected {len(collected)} / {max_reviews} reviews")

        if added == 0:
            raise ValueError("Pagination returned no new review IDs.")

        total = int(pagination["total"])
        actual_page_size = int(pagination["perPage"])

        if (
            len(collected) >= max_reviews
            or page * actual_page_size >= total
        ):
            break

        page += 1
        time.sleep(0.5)

    return collected


cosrx_cleanser_reviews = collect_yotpo_reviews(
    reviews_url=cosrx_cleanser_reviews_url,
    product_id="COSRX_CLEANSER_01",
    max_reviews=300
)

print("\nFinal number of reviews:", len(cosrx_cleanser_reviews))

In [ ]:
def clean_yotpo_review(review):
    fields = {
        field["title"]: field.get("value")
        for field in (review.get("customFields") or {}).values()
        if field.get("title")
    }

    skin_concern = fields.get("Skin Concern")
    skin_concerns = (
        skin_concern if isinstance(skin_concern, list)
        else [skin_concern] if skin_concern else []
    )

    return {
        "review_id": f"yotpo_{review['id']}",
        "product_id": review["relational_product_id"],
        "review_date": review.get("createdAt"),
        "rating": review.get("score"),
        "review_title": review.get("title"),
        "review_text": review.get("content"),
        "reviewer_name": (review.get("user") or {}).get("displayName"),
        "age": fields.get("Age"),
        "skin_type": fields.get("Skin Type"),
        "skin_concerns": skin_concerns,
        "verified": review.get("verifiedBuyer"),
        "recommended": None,  # Not supplied in the inspected response.
        "incentivized": review.get("isIncentivized"),
        "helpful_count": review.get("votesUp"),
        "unhelpful_count": review.get("votesDown"),
        "language": review.get("language"),
        "sponsored": (
            str(fields["Sponsored"]).strip()
            if fields.get("Sponsored") is not None else None
        ),
        "incentive_type": review.get("incentiveType"),
        "grouped_product_name": (
            review.get("groupingData") or {}
        ).get("productName"),
        "variant_size": (
            review.get("productVariants") or {}
        ).get("Size")
    }


cosrx_cleanser_reviews_df = pd.DataFrame([
    clean_yotpo_review(review)
    for review in cosrx_cleanser_reviews
])

# Preserve the source timestamps without assuming a timezone.
cosrx_cleanser_reviews_df["review_date"] = pd.to_datetime(
    cosrx_cleanser_reviews_df["review_date"],
    errors="coerce"
)

print("Reviews:", len(cosrx_cleanser_reviews_df))
print(
    "Duplicate review IDs:",
    cosrx_cleanser_reviews_df["review_id"].duplicated().sum()
)
print(
    "Invalid dates:",
    cosrx_cleanser_reviews_df["review_date"].isna().sum()
)
print(
    "Newest-first order:",
    cosrx_cleanser_reviews_df["review_date"].is_monotonic_decreasing
)

cosrx_cleanser_reviews_df.head()

In [ ]:
cosrx_cleanser_review_skin_concerns_df = (
    cosrx_cleanser_reviews_df[["review_id", "skin_concerns"]]
    .explode("skin_concerns")
    .dropna(subset=["skin_concerns"])
    .rename(columns={"skin_concerns": "skin_concern"})
    .copy()
)

cosrx_cleanser_review_skin_concerns_df["skin_concern"] = (
    cosrx_cleanser_review_skin_concerns_df["skin_concern"]
    .astype(str)
    .str.strip()
)

cosrx_cleanser_review_skin_concerns_df = (
    cosrx_cleanser_review_skin_concerns_df[
        cosrx_cleanser_review_skin_concerns_df["skin_concern"] != ""
    ]
    .drop_duplicates(["review_id", "skin_concern"])
    .reset_index(drop=True)
)

# Preserve all reviews, including those without a reported concern.
cosrx_cleanser_reviews_clean_df = cosrx_cleanser_reviews_df.drop(
    columns=["skin_concerns"]
).copy()

orphan_concerns = (
    ~cosrx_cleanser_review_skin_concerns_df["review_id"].isin(
        cosrx_cleanser_reviews_clean_df["review_id"]
    )
).sum()

assert orphan_concerns == 0, "Orphan review concerns found."

print("Reviews retained:", len(cosrx_cleanser_reviews_clean_df))
print(
    "Review–concern pairs:",
    len(cosrx_cleanser_review_skin_concerns_df)
)
print(
    "Reviews with skin concerns:",
    cosrx_cleanser_review_skin_concerns_df["review_id"].nunique()
)
print("Orphan review concerns:", orphan_concerns)

cosrx_cleanser_review_skin_concerns_df.head(10)

In [ ]:
cosrx_cleanser_config = next(
    product for product in products_config
    if product["product_id"] == "COSRX_CLEANSER_01"
)

cosrx_product_response = requests.get(
    cosrx_cleanser_config["product_url"],
    timeout=30
)
cosrx_product_response.raise_for_status()

cosrx_cleanser_soup = BeautifulSoup(
    cosrx_product_response.text,
    "html.parser"
)

# Inspect all Product blocks before selecting an offer.
cosrx_product_blocks = list_product_blocks(cosrx_cleanser_soup)

print("Product blocks found:", len(cosrx_product_blocks))

for index, product in enumerate(cosrx_product_blocks):
    print(f"\n--- Product block {index} ---")
    print(json.dumps(product, indent=2, ensure_ascii=False))

# Locate possible ingredient sections in the visible page text.
page_text = cosrx_cleanser_soup.get_text(" ", strip=True)

ingredient_matches = list(
    re.finditer(r"\bingredients\b", page_text, flags=re.IGNORECASE)
)

print("\nIngredient mentions found:", len(ingredient_matches))

for match in ingredient_matches[:5]:
    print("\n--- Ingredient section candidate ---")
    print(page_text[max(0, match.start() - 100):match.end() + 2000])

In [ ]:
# Extract the full ingredient section from the downloaded page.
cosrx_page_text = cosrx_cleanser_soup.get_text(" ", strip=True)

ingredient_match = re.search(
    r"Full Ingredients\s+"
    r"(Water,.*?)"
    r"\s+DETAILS\s+HOW TO USE",
    cosrx_page_text,
    flags=re.DOTALL
)

if not ingredient_match:
    raise ValueError("Could not locate the expected full ingredient section.")

cosrx_cleanser_ingredients_raw = ingredient_match.group(1).strip()

ingredient_names = [
    ingredient.strip()
    for ingredient in re.split(
        r",\s+",
        cosrx_cleanser_ingredients_raw
    )
    if ingredient.strip()
]

cosrx_cleanser_ingredients_df = pd.DataFrame({
    "product_id": cosrx_cleanser_config["product_id"],
    "ingredient_position": range(1, len(ingredient_names) + 1),
    "ingredient_name": ingredient_names
})

print("Ingredients:", len(cosrx_cleanser_ingredients_df))
print(
    "1,2-Hexanediol preserved:",
    "1,2-Hexanediol" in ingredient_names
)

cosrx_cleanser_ingredients_df

In [ ]:
from urllib.parse import urlparse, parse_qs

# Select the detailed block with a list of variant offers.
matching_blocks = [
    product for product in cosrx_product_blocks
    if product.get("sku") == "113020478"
    and isinstance(product.get("offers"), list)
]

if len(matching_blocks) != 1:
    raise ValueError("Expected one detailed COSRX product block.")

cosrx_cleanser_product_record = build_product_record(
    cosrx_cleanser_config,
    matching_blocks[0]
)

# Confirm the selected offer is the user-verified 150 ml variant.
variant_id = parse_qs(
    urlparse(cosrx_cleanser_product_record["variant_url"]).query
).get("variant", [None])[0]

assert variant_id == "37517677953180", "Unexpected product variant."
assert cosrx_cleanser_product_record["current_price"] == 14.0
assert cosrx_cleanser_product_record["currency"] == "USD"

cosrx_cleanser_product_record.update({
    "size_ml": 150,
    "key_ingredients": "Betaine Salicylate (BHA) & Tea Tree Oil",
    "key_benefits": (
        "Hydrating cleanse without stripping; Removes impurities; "
        "Protects skin barrier; Restores pH Balance"
    ),
    "skin_concern": (
        "Dry skin; Dull & tired skin; Uneven skin texture"
    )
})

cosrx_cleanser_products_df = pd.DataFrame([
    cosrx_cleanser_product_record
])

# Validate the four tables before saving.
product_id = "COSRX_CLEANSER_01"
reviews = cosrx_cleanser_reviews_clean_df
ingredients = cosrx_cleanser_ingredients_df
concerns = cosrx_cleanser_review_skin_concerns_df

assert len(reviews) == 300
assert reviews["review_id"].notna().all()
assert reviews["review_id"].is_unique
assert reviews["review_date"].notna().all()
assert reviews["review_date"].is_monotonic_decreasing
assert reviews["product_id"].eq(product_id).all()
assert len(ingredients) == 25
assert ingredients["product_id"].eq(product_id).all()
assert concerns["review_id"].isin(reviews["review_id"]).all()
assert not concerns.duplicated(["review_id", "skin_concern"]).any()

# Save without recollecting reviews.
cosrx_cleanser_tables = {
    "products": cosrx_cleanser_products_df,
    "reviews": reviews,
    "ingredients": ingredients,
    "review_skin_concerns": concerns
}

output_folder = Path("data/processed") / product_id
output_folder.mkdir(parents=True, exist_ok=True)

for name, table in cosrx_cleanser_tables.items():
    file_path = output_folder / f"{name}.csv"
    table.to_csv(file_path, index=False)

    assert len(pd.read_csv(file_path)) == len(table)
    print(f"{name}.csv: {len(table)} rows saved")

print("\nAll checks passed.")
print("Saved to:", output_folder.resolve())

In [ ]:
cosrx_cream_config = next(
    product for product in products_config
    if product["product_id"] == "COSRX_MOISTURIZER_01"
)

cosrx_cream_response = requests.get(
    cosrx_cream_config["product_url"],
    timeout=30
)
cosrx_cream_response.raise_for_status()

cosrx_cream_soup = BeautifulSoup(
    cosrx_cream_response.text,
    "html.parser"
)

cosrx_cream_product_blocks = list_product_blocks(cosrx_cream_soup)

print("Product blocks found:", len(cosrx_cream_product_blocks))

for index, product in enumerate(cosrx_cream_product_blocks):
    print(f"\n--- Product block {index} ---")
    print(json.dumps(product, indent=2, ensure_ascii=False))

# Show the full ingredient section if the same layout is present.
cream_page_text = cosrx_cream_soup.get_text(" ", strip=True)

ingredient_sections = list(re.finditer(
    r"Full Ingredients",
    cream_page_text,
    flags=re.IGNORECASE
))

print("\n--- Ingredient section ---")

if ingredient_sections:
    # The first occurrence may be the button opening the ingredient list.
    start = ingredient_sections[0].start()
    print(cream_page_text[start:start + 4000])
else:
    print("Full Ingredients heading not found.")

In [ ]:
cosrx_cream_reviews_url = (
    "https://api-cdn.yotpo.com/v3/storefront/store/"
    "FHrLUCZg4k9lvDprUIkr92ql2rzsUDWvqR67HEcq/"
    "product/9072734372056/reviews"
)

cosrx_cream_reviews = collect_yotpo_reviews(
    reviews_url=cosrx_cream_reviews_url,
    product_id="COSRX_MOISTURIZER_01",
    max_reviews=300
)

cosrx_cream_reviews_df = pd.DataFrame([
    clean_yotpo_review(review)
    for review in cosrx_cream_reviews
])

cosrx_cream_reviews_df["review_date"] = pd.to_datetime(
    cosrx_cream_reviews_df["review_date"],
    errors="coerce"
)

# Create the separate review–skin concern table.
cosrx_cream_review_skin_concerns_df = (
    cosrx_cream_reviews_df[["review_id", "skin_concerns"]]
    .explode("skin_concerns")
    .dropna(subset=["skin_concerns"])
    .rename(columns={"skin_concerns": "skin_concern"})
    .copy()
)

concerns = cosrx_cream_review_skin_concerns_df
concerns["skin_concern"] = (
    concerns["skin_concern"].astype(str).str.strip()
)

cosrx_cream_review_skin_concerns_df = (
    concerns[concerns["skin_concern"] != ""]
    .drop_duplicates(["review_id", "skin_concern"])
    .reset_index(drop=True)
)

cosrx_cream_reviews_clean_df = cosrx_cream_reviews_df.drop(
    columns=["skin_concerns"]
).copy()

# Validate before saving.
reviews = cosrx_cream_reviews_clean_df
concerns = cosrx_cream_review_skin_concerns_df

assert reviews["review_id"].notna().all()
assert reviews["review_id"].is_unique
assert reviews["review_date"].notna().all()
assert reviews["review_date"].is_monotonic_decreasing
assert concerns["review_id"].isin(reviews["review_id"]).all()

print("\nReviews:", len(reviews))
print("Review–concern pairs:", len(concerns))
print("Reviews with skin concerns:", concerns["review_id"].nunique())
print("All review checks passed.")

In [ ]:
# Select the detailed product block matching the confirmed jar SKU.
cream_matches = [
    product for product in cosrx_cream_product_blocks
    if product.get("sku") == "113020301"
    and isinstance(product.get("offers"), list)
]

if len(cream_matches) != 1:
    raise ValueError("Expected one detailed cream product block.")

cosrx_cream_product_record = build_product_record(
    cosrx_cream_config,
    cream_matches[0]
)

from urllib.parse import urlparse, parse_qs

variant_id = parse_qs(
    urlparse(cosrx_cream_product_record["variant_url"]).query
).get("variant", [None])[0]

assert variant_id == "51096813043928", "Unexpected variant."
assert cosrx_cream_product_record["current_price"] == 26.0
assert cosrx_cream_product_record["currency"] == "USD"

cosrx_cream_product_record.update({
    "size_g": 100,
    "packaging": "Jar",
    "key_ingredients": (
        "92% Snail Secretion Filtrate; Allantoin & Panthenol; "
        "Hyaluronic Acid & Betaine"
    ),
    "key_benefits": (
        "Boosts skin radiance and visibly enhances skin clarity; "
        "Soothes the appearance of irritated skin; "
        "Provides long-lasting and deep hydration"
    ),
    "skin_concern": None,  # Dedicated concerns section not yet captured.
    "ingredients_source": (
        "Manually transcribed from COSRX Ingredient List screenshot "
        "provided by user, 2026-09-28"
    )
})

cosrx_cream_products_df = pd.DataFrame([
    cosrx_cream_product_record
])

# Preserve the ingredient names and order displayed on the page.
cosrx_cream_ingredient_names = [
    "Snail Secretion Filtrate",
    "Betaine",
    "Caprylic/Capric Triglyceride",
    "Butylene Glycol",
    "Cetearyl Olivate",
    "Sorbitan Olivate",
    "Cetearyl Alcohol",
    "Carbomer",
    "Ethyl Hexanediol",
    "Phenoxyethanol",
    "Arginine",
    "Dimethicone",
    "Sodium Polyacrylate",
    "Sodium Hyaluronate",
    "Allantoin",
    "Palmitic Acid",
    "Panthenol",
    "Xanthan Gum",
    "Stearic Acid",
    "Adenosine",
    "Water",
    "Myristic Acid"
]

cosrx_cream_ingredients_df = pd.DataFrame({
    "product_id": cosrx_cream_config["product_id"],
    "ingredient_position": range(
        1, len(cosrx_cream_ingredient_names) + 1
    ),
    "ingredient_name": cosrx_cream_ingredient_names
})

print("Product:", cosrx_cream_product_record["product_name"])
print("Size / format: 100 g jar")
print("Price: 26.00 USD")
print("Ingredients:", len(cosrx_cream_ingredients_df))

cosrx_cream_ingredients_df

In [ ]:
from pathlib import Path

required_tables = [
    "cosrx_cream_reviews_clean_df",
    "cosrx_cream_review_skin_concerns_df"
]

missing = [
    name for name in required_tables
    if name not in globals()
]

if missing:
    raise ValueError(
        "Run the COSRX cream review-collection and cleaning cell first. "
        f"Missing: {', '.join(missing)}"
    )

product_id = "COSRX_MOISTURIZER_01"

products = cosrx_cream_products_df
reviews = cosrx_cream_reviews_clean_df
ingredients = cosrx_cream_ingredients_df
concerns = cosrx_cream_review_skin_concerns_df

# Validate before saving.
assert products["product_id"].tolist() == [product_id]
assert len(reviews) == 300, "Expected 300 cream reviews."
assert reviews["review_id"].notna().all()
assert reviews["review_id"].is_unique
assert reviews["product_id"].eq(product_id).all()
assert reviews["review_date"].notna().all()
assert reviews["review_date"].is_monotonic_decreasing
assert len(ingredients) == 22
assert ingredients["product_id"].eq(product_id).all()
assert concerns["review_id"].isin(reviews["review_id"]).all()
assert not concerns.duplicated(["review_id", "skin_concern"]).any()

cosrx_cream_tables = {
    "products": products,
    "reviews": reviews,
    "ingredients": ingredients,
    "review_skin_concerns": concerns
}

output_folder = Path("data/processed") / product_id
output_folder.mkdir(parents=True, exist_ok=True)

for name, table in cosrx_cream_tables.items():
    file_path = output_folder / f"{name}.csv"
    table.to_csv(file_path, index=False)

    assert len(pd.read_csv(file_path)) == len(table)
    print(f"{name}.csv: {len(table)} rows saved")

print("\nAll checks passed.")
print("Saved to:", output_folder.resolve())

In [ ]:
cosrx_serum_config = next(
    product for product in products_config
    if product["product_id"] == "COSRX_SERUM_01"
)

cosrx_serum_response = requests.get(
    cosrx_serum_config["product_url"],
    timeout=30
)
cosrx_serum_response.raise_for_status()

cosrx_serum_soup = BeautifulSoup(
    cosrx_serum_response.text,
    "html.parser"
)

print("Final page URL:", cosrx_serum_response.url)

cosrx_serum_product_blocks = list_product_blocks(cosrx_serum_soup)

print("Product blocks found:", len(cosrx_serum_product_blocks))

for index, product in enumerate(cosrx_serum_product_blocks):
    print(f"\n--- Product block {index} ---")
    print(json.dumps(product, indent=2, ensure_ascii=False))

# Check both ingredient headings encountered on COSRX.
serum_page_text = cosrx_serum_soup.get_text(" ", strip=True)

ingredient_matches = list(re.finditer(
    r"Full Ingredients|Ingredient List",
    serum_page_text,
    flags=re.IGNORECASE
))

print("\n--- Ingredient section ---")

if ingredient_matches:
    start = ingredient_matches[0].start()
    print(serum_page_text[start:start + 4500])
else:
    print("Ingredient heading not found in the downloaded page.")

In [ ]:
cosrx_serum_reviews_url = (
    "https://api-cdn.yotpo.com/v3/storefront/store/"
    "FHrLUCZg4k9lvDprUIkr92ql2rzsUDWvqR67HEcq/"
    "product/9394648187096/reviews"
)

# Starts at page 1 and checks newest-first ordering.
cosrx_serum_reviews = collect_yotpo_reviews(
    reviews_url=cosrx_serum_reviews_url,
    product_id="COSRX_SERUM_01",
    max_reviews=300
)

cosrx_serum_reviews_df = pd.DataFrame([
    clean_yotpo_review(review)
    for review in cosrx_serum_reviews
])

cosrx_serum_reviews_df["review_date"] = pd.to_datetime(
    cosrx_serum_reviews_df["review_date"],
    errors="coerce"
)

# Separate reviewer skin concerns.
cosrx_serum_review_skin_concerns_df = (
    cosrx_serum_reviews_df[["review_id", "skin_concerns"]]
    .explode("skin_concerns")
    .dropna(subset=["skin_concerns"])
    .rename(columns={"skin_concerns": "skin_concern"})
    .copy()
)

concerns = cosrx_serum_review_skin_concerns_df
concerns["skin_concern"] = (
    concerns["skin_concern"].astype(str).str.strip()
)

cosrx_serum_review_skin_concerns_df = (
    concerns[concerns["skin_concern"] != ""]
    .drop_duplicates(["review_id", "skin_concern"])
    .reset_index(drop=True)
)

cosrx_serum_reviews_clean_df = cosrx_serum_reviews_df.drop(
    columns=["skin_concerns"]
).copy()

reviews = cosrx_serum_reviews_clean_df
concerns = cosrx_serum_review_skin_concerns_df

assert reviews["review_id"].notna().all()
assert reviews["review_id"].is_unique
assert reviews["review_date"].notna().all()
assert reviews["review_date"].is_monotonic_decreasing
assert reviews["product_id"].eq("COSRX_SERUM_01").all()
assert concerns["review_id"].isin(reviews["review_id"]).all()

print("\nReviews:", len(reviews))
print("Review–concern pairs:", len(concerns))
print("Reviews with skin concerns:", concerns["review_id"].nunique())
print("All review checks passed.")

In [ ]:
from urllib.parse import urlparse, parse_qs

# Find the exact offer confirmed by your screenshot.
selected_variant = "52315200258264"
matches = []

for product in cosrx_serum_product_blocks:
    offers = product.get("offers", [])
    if isinstance(offers, dict):
        offers = [offers]

    for offer in offers:
        variant = parse_qs(
            urlparse(offer.get("url", "")).query
        ).get("variant", [None])[0]

        if variant == selected_variant:
            matches.append((product, offer))

if len(matches) != 1:
    raise ValueError("Expected one offer for the selected serum variant.")

serum_product, serum_offer = matches[0]

assert float(serum_offer["price"]) == 25.0
assert serum_offer["priceCurrency"] == "USD"

cosrx_serum_product_record = {
    "product_id": cosrx_serum_config["product_id"],
    "brand": cosrx_serum_config["brand"],
    "product_name": serum_product["name"],
    "segment": cosrx_serum_config["segment"],
    "sku": serum_offer.get("sku"),
    "current_price": float(serum_offer["price"]),
    "currency": serum_offer["priceCurrency"],
    "description": (serum_product.get("description") or "").strip(),
    "product_url": cosrx_serum_config["product_url"],
    "variant_url": serum_offer["url"],
    "size_ml": 150,
    "pack_quantity": 1,
    "key_ingredients": (
        "6 peptide complex; Natural Moisturizing Factors & "
        "Hyaluronic Acids; Niacinamide & Adenosine"
    ),
    "key_benefits": (
        "Firmness; Radiance; Smoothness; Deep hydration; Glow"
    ),
    "skin_concern": (
        "Enlarged pores; Fine lines; Dullness; "
        "Dehydration; Uneven skin tone"
    )
}

cosrx_serum_products_df = pd.DataFrame([
    cosrx_serum_product_record
])

# Extract the ingredient section from the downloaded page.
serum_page_text = cosrx_serum_soup.get_text(" ", strip=True)

ingredient_match = re.search(
    r"Ingredient List\s+(Water,.*?)\s*🤖",
    serum_page_text,
    flags=re.DOTALL
)

if not ingredient_match:
    raise ValueError("Could not locate the expected ingredient section.")

cosrx_serum_ingredients_raw = ingredient_match.group(1).strip()

serum_ingredient_names = [
    ingredient.strip()
    for ingredient in re.split(r",\s+", cosrx_serum_ingredients_raw)
    if ingredient.strip()
]

cosrx_serum_ingredients_df = pd.DataFrame({
    "product_id": cosrx_serum_config["product_id"],
    "ingredient_position": range(1, len(serum_ingredient_names) + 1),
    "ingredient_name": serum_ingredient_names
})

print("Product:", cosrx_serum_product_record["product_name"])
print("Size / quantity: 150 ml / 1 bottle")
print("Price: 25.00 USD")
print("Ingredients:", len(cosrx_serum_ingredients_df))
print("1,2-Hexanediol preserved:", "1,2-Hexanediol" in serum_ingredient_names)

cosrx_serum_ingredients_df

In [ ]:
from pathlib import Path

required = [
    "cosrx_serum_reviews_clean_df",
    "cosrx_serum_review_skin_concerns_df"
]

missing = [name for name in required if name not in globals()]

if missing:
    raise ValueError(
        "Run the serum review-collection and cleaning cell first. "
        f"Missing: {', '.join(missing)}"
    )

product_id = "COSRX_SERUM_01"

products = cosrx_serum_products_df
reviews = cosrx_serum_reviews_clean_df
ingredients = cosrx_serum_ingredients_df
concerns = cosrx_serum_review_skin_concerns_df

# Validate before saving.
assert products["product_id"].tolist() == [product_id]
assert 0 < len(reviews) <= 300, "Expected 1–300 reviews."
assert reviews["review_id"].notna().all()
assert reviews["review_id"].is_unique
assert reviews["product_id"].eq(product_id).all()
assert reviews["review_date"].notna().all()
assert reviews["review_date"].is_monotonic_decreasing
assert len(ingredients) == 44
assert ingredients["product_id"].eq(product_id).all()
assert concerns["review_id"].isin(reviews["review_id"]).all()
assert not concerns.duplicated(["review_id", "skin_concern"]).any()

cosrx_serum_tables = {
    "products": products,
    "reviews": reviews,
    "ingredients": ingredients,
    "review_skin_concerns": concerns
}

output_folder = Path("data/processed") / product_id
output_folder.mkdir(parents=True, exist_ok=True)

for name, table in cosrx_serum_tables.items():
    file_path = output_folder / f"{name}.csv"
    table.to_csv(file_path, index=False)

    assert len(pd.read_csv(file_path)) == len(table)
    print(f"{name}.csv: {len(table)} rows saved")

print("\nAll checks passed.")
print("Saved to:", output_folder.resolve())

In [ ]:
cosrx_toner_config = next(
    product for product in products_config
    if product["product_id"] == "COSRX_TONER_01"
)

cosrx_toner_response = requests.get(
    cosrx_toner_config["product_url"],
    timeout=30
)
cosrx_toner_response.raise_for_status()

cosrx_toner_soup = BeautifulSoup(
    cosrx_toner_response.text,
    "html.parser"
)

print("Final page URL:", cosrx_toner_response.url)

cosrx_toner_product_blocks = list_product_blocks(cosrx_toner_soup)

print("Product blocks found:", len(cosrx_toner_product_blocks))

for index, product in enumerate(cosrx_toner_product_blocks):
    print(f"\n--- Product block {index} ---")
    print(json.dumps(product, indent=2, ensure_ascii=False))

toner_page_text = cosrx_toner_soup.get_text(" ", strip=True)

ingredient_matches = list(re.finditer(
    r"Full Ingredients|Ingredient List",
    toner_page_text,
    flags=re.IGNORECASE
))

print("\n--- Ingredient section ---")

if ingredient_matches:
    start = ingredient_matches[0].start()
    print(toner_page_text[start:start + 3500])
else:
    print("Ingredient heading not found in the downloaded page.")

In [ ]:
from pathlib import Path
from urllib.parse import urlparse, parse_qs

# 1. Select the detailed product block.
matches = [
    product for product in cosrx_toner_product_blocks
    if product.get("sku") == "113020124"
    and isinstance(product.get("offers"), list)
]

if len(matches) != 1:
    raise ValueError("Expected one detailed toner product block.")

cosrx_toner_product_record = build_product_record(
    cosrx_toner_config,
    matches[0]
)

variant_id = parse_qs(
    urlparse(cosrx_toner_product_record["variant_url"]).query
).get("variant", [None])[0]

assert variant_id == "43487462129880", "Unexpected variant."
assert cosrx_toner_product_record["current_price"] == 29.0
assert cosrx_toner_product_record["currency"] == "USD"

cosrx_toner_product_record.update({
    "size_ml": 280,
    "key_ingredients": "Propolis Extract; Honey Extract; Panthenol",
    "key_benefits": (
        "Nourish for healthy skin; Plump, glowing skin; "
        "Reduce redness; Improve skin texture"
    ),
    "skin_concern": "Dull, dehydrated skin; Red, irritated skin"
})

cosrx_toner_products_df = pd.DataFrame([
    cosrx_toner_product_record
])

# 2. Extract the full ingredient list from the downloaded page.
toner_page_text = cosrx_toner_soup.get_text(" ", strip=True)

ingredient_match = re.search(
    r"Full Ingredients\s+(Propolis Extract,.*?)"
    r"\s+DETAILS\s+HOW TO USE",
    toner_page_text,
    flags=re.DOTALL
)

if not ingredient_match:
    raise ValueError("Could not locate the full ingredient section.")

cosrx_toner_ingredients_raw = ingredient_match.group(1).strip()

ingredient_names = [
    ingredient.strip()
    for ingredient in re.split(r",\s+", cosrx_toner_ingredients_raw)
    if ingredient.strip()
]

assert len(ingredient_names) == 14, "Unexpected ingredient count."
assert "1,2-Hexanediol" in ingredient_names

cosrx_toner_ingredients_df = pd.DataFrame({
    "product_id": cosrx_toner_config["product_id"],
    "ingredient_position": range(1, len(ingredient_names) + 1),
    "ingredient_name": ingredient_names
})

# 3. Collect reviews from page 1 with the verified sorting.
cosrx_toner_reviews_url = (
    "https://api-cdn.yotpo.com/v3/storefront/store/"
    "FHrLUCZg4k9lvDprUIkr92ql2rzsUDWvqR67HEcq/"
    "product/6101924348060/reviews"
)

cosrx_toner_reviews = collect_yotpo_reviews(
    reviews_url=cosrx_toner_reviews_url,
    product_id="COSRX_TONER_01",
    max_reviews=300
)

if not cosrx_toner_reviews:
    raise ValueError("No toner reviews returned.")

cosrx_toner_reviews_df = pd.DataFrame([
    clean_yotpo_review(review)
    for review in cosrx_toner_reviews
])

cosrx_toner_reviews_df["review_date"] = pd.to_datetime(
    cosrx_toner_reviews_df["review_date"],
    errors="coerce"
)

# 4. Separate reviewer skin concerns.
concerns = (
    cosrx_toner_reviews_df[["review_id", "skin_concerns"]]
    .explode("skin_concerns")
    .dropna(subset=["skin_concerns"])
    .rename(columns={"skin_concerns": "skin_concern"})
    .copy()
)

concerns["skin_concern"] = (
    concerns["skin_concern"].astype(str).str.strip()
)

cosrx_toner_review_skin_concerns_df = (
    concerns[concerns["skin_concern"] != ""]
    .drop_duplicates(["review_id", "skin_concern"])
    .reset_index(drop=True)
)

cosrx_toner_reviews_clean_df = cosrx_toner_reviews_df.drop(
    columns=["skin_concerns"]
).copy()

# 5. Validate before saving.
product_id = "COSRX_TONER_01"
reviews = cosrx_toner_reviews_clean_df
concerns = cosrx_toner_review_skin_concerns_df

assert cosrx_toner_products_df["product_id"].tolist() == [product_id]
assert reviews["review_id"].notna().all()
assert reviews["review_id"].is_unique
assert reviews["product_id"].eq(product_id).all()
assert reviews["review_date"].notna().all()
assert reviews["review_date"].is_monotonic_decreasing
assert cosrx_toner_ingredients_df["product_id"].eq(product_id).all()
assert concerns["review_id"].isin(reviews["review_id"]).all()
assert not concerns.duplicated(["review_id", "skin_concern"]).any()

# 6. Save the four tables.
cosrx_toner_tables = {
    "products": cosrx_toner_products_df,
    "reviews": reviews,
    "ingredients": cosrx_toner_ingredients_df,
    "review_skin_concerns": concerns
}

output_folder = Path("data/processed") / product_id
output_folder.mkdir(parents=True, exist_ok=True)

for name, table in cosrx_toner_tables.items():
    file_path = output_folder / f"{name}.csv"
    table.to_csv(file_path, index=False)
    assert len(pd.read_csv(file_path)) == len(table)
    print(f"{name}.csv: {len(table)} rows saved")

print("\nAll checks passed.")
print("Saved to:", output_folder.resolve())

In [ ]:
from pathlib import Path

cosrx_product_ids = [
    "COSRX_CLEANSER_01",
    "COSRX_MOISTURIZER_01",
    "COSRX_SERUM_01",
    "COSRX_TONER_01"
]

table_names = [
    "products",
    "reviews",
    "ingredients",
    "review_skin_concerns"
]

# Check that every product's files were saved.
required_files = [
    Path("data/processed") / product_id / f"{name}.csv"
    for product_id in cosrx_product_ids
    for name in table_names
]

missing = [str(path) for path in required_files if not path.exists()]

if missing:
    raise ValueError("Missing product files:\n" + "\n".join(missing))

# Combine only the four product folders.
cosrx_tables = {}

for name in table_names:
    cosrx_tables[name] = pd.concat(
        [
            pd.read_csv(
                Path("data/processed") / product_id / f"{name}.csv",
                dtype={"product_id": str, "review_id": str, "sku": str}
            )
            for product_id in cosrx_product_ids
        ],
        ignore_index=True
    )

products = cosrx_tables["products"]
reviews = cosrx_tables["reviews"]
ingredients = cosrx_tables["ingredients"]
concerns = cosrx_tables["review_skin_concerns"]

# Validate relationships and duplicate keys.
assert products["product_id"].is_unique
assert set(products["product_id"]) == set(cosrx_product_ids)
assert reviews["review_id"].notna().all()
assert reviews["review_id"].is_unique, (
    "Review IDs overlap across products. Inspect before combining."
)
assert reviews["product_id"].isin(products["product_id"]).all()
assert ingredients["product_id"].isin(products["product_id"]).all()
assert concerns["review_id"].isin(reviews["review_id"]).all()
assert not ingredients.duplicated(
    ["product_id", "ingredient_position"]
).any()
assert not concerns.duplicated(["review_id", "skin_concern"]).any()

# Save combined brand files.
output_folder = Path("data/processed/cosrx")
output_folder.mkdir(parents=True, exist_ok=True)

for name, table in cosrx_tables.items():
    file_path = output_folder / f"{name}.csv"
    table.to_csv(file_path, index=False)
    assert len(pd.read_csv(file_path)) == len(table)
    print(f"{name}.csv: {len(table)} rows saved")

print("\nReviews per product:")
print(reviews.groupby("product_id").size().to_string())

print("\nAll combined checks passed.")
print("Saved to:", output_folder.resolve())

In [ ]:
laneige_cleanser_reviews_url = (
    "https://api-cdn.yotpo.com/v3/storefront/store/"
    "pc9NUmcX8nZOBN32faV43yrNlBhSI07Q5DPJm1uE/"
    "product/7704425562164/reviews"
)

laneige_reviews_response = requests.get(
    laneige_cleanser_reviews_url,
    params={
        "page": 1,
        "perPage": 5,
        "sort": "date,badge,images,rating"
    },
    timeout=30
)
laneige_reviews_response.raise_for_status()

laneige_reviews_data = laneige_reviews_response.json()

print("Pagination:")
print(json.dumps(laneige_reviews_data.get("pagination"), indent=2))

print("\nProducts:")
print(json.dumps(
    laneige_reviews_data.get("products"),
    indent=2,
    ensure_ascii=False
))

preview_reviews = laneige_reviews_data.get("reviews", [])

print("\nFirst review:")
if preview_reviews:
    print(json.dumps(
        preview_reviews[0],
        indent=2,
        ensure_ascii=False
    ))
else:
    print("No reviews returned.")

print("\nReview dates:")
for review in preview_reviews:
    print(review.get("createdAt"))

In [ ]:
def collect_laneige_reviews(reviews_url, product_id, max_reviews=300):
    collected = []
    seen_ids = set()
    page = 1
    per_page = 5
    previous_date = None

    while len(collected) < max_reviews:
        response = requests.get(
            reviews_url,
            params={
                "page": page,
                "perPage": per_page,
                "sort": "date,badge,images,rating"
            },
            timeout=30
        )
        response.raise_for_status()
        data = response.json()

        pagination = data["pagination"]
        reviews = data["reviews"]

        if int(pagination["page"]) != page:
            raise ValueError("Unexpected page returned.")

        if not reviews:
            break

        added = 0

        for review in reviews:
            review_id = review.get("id")
            if review_id is None:
                raise ValueError("Missing review ID.")

            if review_id in seen_ids:
                continue

            review_date = pd.to_datetime(
                review.get("createdAt"), errors="coerce"
            )

            if pd.isna(review_date):
                raise ValueError("Missing or invalid review date.")

            if previous_date is not None and review_date > previous_date:
                raise ValueError(f"Newest-first ordering failed on page {page}.")

            previous_date = review_date
            seen_ids.add(review_id)
            collected.append({
                **review,
                "relational_product_id": product_id
            })
            added += 1

            if len(collected) >= max_reviews:
                break

        if added == 0:
            raise ValueError("Pagination returned no new reviews.")

        if page % 5 == 0 or len(collected) >= max_reviews:
            print(f"Collected {len(collected)} / {max_reviews} reviews")

        if (
            len(collected) >= max_reviews
            or page * int(pagination["perPage"]) >= int(pagination["total"])
        ):
            break

        page += 1
        time.sleep(0.5)

    return collected


def clean_laneige_review(review):
    # Reuse shared Yotpo fields, then adapt LANEIGE's custom fields.
    record = clean_yotpo_review(review)

    fields = {
        field["title"].strip().casefold(): field.get("value")
        for field in (review.get("customFields") or {}).values()
        if field.get("title")
    }

    record["age"] = fields.get("age")
    record["skin_type"] = fields.get("skin type")
    record["skincare_goal"] = fields.get("skincare goal")

    concern = fields.get("skin concern", fields.get("skin concerns"))
    record["skin_concerns"] = (
        concern if isinstance(concern, list)
        else [concern] if concern else []
    )

    return record


laneige_cleanser_reviews = collect_laneige_reviews(
    reviews_url=laneige_cleanser_reviews_url,
    product_id="LANEIGE_CLEANSER_01",
    max_reviews=300
)

laneige_cleanser_reviews_df = pd.DataFrame([
    clean_laneige_review(review)
    for review in laneige_cleanser_reviews
])

laneige_cleanser_reviews_df["review_date"] = pd.to_datetime(
    laneige_cleanser_reviews_df["review_date"],
    errors="coerce"
)

print("\nReviews:", len(laneige_cleanser_reviews_df))
print(
    "Duplicate review IDs:",
    laneige_cleanser_reviews_df["review_id"].duplicated().sum()
)
print(
    "Invalid dates:",
    laneige_cleanser_reviews_df["review_date"].isna().sum()
)
print(
    "Newest-first order:",
    laneige_cleanser_reviews_df["review_date"].is_monotonic_decreasing
)
print(
    "Reviews with skincare goals:",
    laneige_cleanser_reviews_df["skincare_goal"].notna().sum()
)

In [ ]:
from collections import Counter

laneige_cleanser_config = next(
    product for product in products_config
    if product["product_id"] == "LANEIGE_CLEANSER_01"
)

laneige_product_response = requests.get(
    laneige_cleanser_config["product_url"],
    timeout=30
)
laneige_product_response.raise_for_status()

laneige_cleanser_soup = BeautifulSoup(
    laneige_product_response.text,
    "html.parser"
)

laneige_cleanser_product_blocks = list_product_blocks(
    laneige_cleanser_soup
)

print("Product blocks found:", len(laneige_cleanser_product_blocks))

for index, product in enumerate(laneige_cleanser_product_blocks):
    print(f"\n--- Product block {index} ---")
    print(json.dumps(product, indent=2, ensure_ascii=False))

# Locate ingredient text without assuming COSRX's page layout.
laneige_page_text = laneige_cleanser_soup.get_text(" ", strip=True)

ingredient_matches = list(re.finditer(
    r"\bingredients\b|\bingredient list\b",
    laneige_page_text,
    flags=re.IGNORECASE
))

print("\nIngredient mentions found:", len(ingredient_matches))

for match in ingredient_matches[:4]:
    print("\n--- Ingredient section candidate ---")
    print(laneige_page_text[
        max(0, match.start() - 80):match.end() + 2500
    ])

# Check field labels across all 300 reviews.
field_counts = Counter(
    field["title"].strip()
    for review in laneige_cleanser_reviews
    for field in (review.get("customFields") or {}).values()
    if field.get("title")
)

print("\nReviewer field labels and occurrence counts:")
for title, count in field_counts.most_common():
    print(f"{title}: {count}")

In [ ]:
from pathlib import Path
from urllib.parse import urlparse, parse_qs

# Select the confirmed 250 ml variant.
matches = [
    product for product in laneige_cleanser_product_blocks
    if parse_qs(
        urlparse(product.get("url", "")).query
    ).get("variant", [None])[0] == "45866099671092"
]

if len(matches) != 1:
    raise ValueError("Expected one Product block for the 250 ml variant.")

product = matches[0]

laneige_cleanser_product_record = {
    "product_id": laneige_cleanser_config["product_id"],
    "brand": laneige_cleanser_config["brand"],
    "product_name": laneige_cleanser_config["product_name"],
    "segment": laneige_cleanser_config["segment"],
    "sku": product["sku"],
    "current_price": 29.0,
    "currency": "USD",
    "size_ml": 250,
    "purchase_type": "One-time purchase",
    "price_source": "User-provided product-page screenshot, 2026-09-28",
    "description": product.get("description", "").strip(),
    "product_url": laneige_cleanser_config["product_url"],
    "variant_url": product["url"],
    "key_ingredients": (
        "Blue Hyaluronic Acid; Allantoin; Amino-acid Complex"
    ),
    "key_benefits": (
        "Removes impurities, face makeup and sunscreen; "
        "Hydrates; Soothes; Supports the moisture barrier"
    ),
    "skin_concern": None
}

laneige_cleanser_products_df = pd.DataFrame([
    laneige_cleanser_product_record
])

# Extract the full formula directly from the structured product data.
properties = product.get("additionalProperty", [])
if isinstance(properties, dict):
    properties = [properties]

ingredient_values = [
    item["value"]
    for item in properties
    if item.get("name", "").casefold() == "ingredients"
    and isinstance(item.get("value"), str)
]

if len(ingredient_values) != 1:
    raise ValueError("Expected one full ingredient list.")

laneige_cleanser_ingredients_raw = ingredient_values[0]

ingredient_names = [
    name.strip()
    for name in re.split(r",\s+", laneige_cleanser_ingredients_raw)
    if name.strip()
]

assert len(ingredient_names) == 27
assert "1,2-HEXANEDIOL" in ingredient_names

laneige_cleanser_ingredients_df = pd.DataFrame({
    "product_id": laneige_cleanser_config["product_id"],
    "ingredient_position": range(1, len(ingredient_names) + 1),
    "ingredient_name": ingredient_names
})

# Keep goals separate from concerns.
concerns = (
    laneige_cleanser_reviews_df[["review_id", "skin_concerns"]]
    .explode("skin_concerns")
    .dropna(subset=["skin_concerns"])
    .rename(columns={"skin_concerns": "skin_concern"})
    .copy()
)

concerns["skin_concern"] = (
    concerns["skin_concern"].astype(str).str.strip()
)

laneige_cleanser_review_skin_concerns_df = (
    concerns[concerns["skin_concern"] != ""]
    .drop_duplicates(["review_id", "skin_concern"])
    .reset_index(drop=True)
)

laneige_cleanser_reviews_clean_df = laneige_cleanser_reviews_df.drop(
    columns=["skin_concerns"]
).copy()

# Validate before saving.
reviews = laneige_cleanser_reviews_clean_df
concerns = laneige_cleanser_review_skin_concerns_df
product_id = "LANEIGE_CLEANSER_01"

assert laneige_cleanser_products_df["product_id"].tolist() == [product_id]
assert len(reviews) == 300
assert reviews["review_id"].notna().all()
assert reviews["review_id"].is_unique
assert reviews["product_id"].eq(product_id).all()
assert reviews["review_date"].notna().all()
assert reviews["review_date"].is_monotonic_decreasing
assert laneige_cleanser_ingredients_df["product_id"].eq(product_id).all()
assert concerns["review_id"].isin(reviews["review_id"]).all()

laneige_cleanser_tables = {
    "products": laneige_cleanser_products_df,
    "reviews": reviews,
    "ingredients": laneige_cleanser_ingredients_df,
    "review_skin_concerns": concerns
}

output_folder = Path("data/processed") / product_id
output_folder.mkdir(parents=True, exist_ok=True)

for name, table in laneige_cleanser_tables.items():
    file_path = output_folder / f"{name}.csv"
    table.to_csv(file_path, index=False)
    assert len(pd.read_csv(file_path)) == len(table)
    print(f"{name}.csv: {len(table)} rows saved")

print("\nAll checks passed.")
print("Saved to:", output_folder.resolve())

In [ ]:
laneige_cream_config = next(
    product for product in products_config
    if product["product_id"] == "LANEIGE_MOISTURIZER_01"
)

laneige_cream_response = requests.get(
    laneige_cream_config["product_url"],
    timeout=30
)
laneige_cream_response.raise_for_status()

laneige_cream_soup = BeautifulSoup(
    laneige_cream_response.text,
    "html.parser"
)

laneige_cream_product_blocks = list_product_blocks(
    laneige_cream_soup
)

print("Final page URL:", laneige_cream_response.url)
print("Product blocks found:", len(laneige_cream_product_blocks))

for index, product in enumerate(laneige_cream_product_blocks):
    print(f"\n--- Product block {index} ---")
    print(json.dumps(product, indent=2, ensure_ascii=False))

In [ ]:
from urllib.parse import urlparse, parse_qs
from collections import Counter

# 1. Select the Matin Kim variant.
matches = [
    product for product in laneige_cream_product_blocks
    if parse_qs(
        urlparse(product.get("url", "")).query
    ).get("variant", [None])[0] == "48218677739572"
]

if len(matches) != 1:
    raise ValueError("Expected one Matin Kim 50 ml product block.")

product = matches[0]

laneige_cream_product_record = {
    "product_id": laneige_cream_config["product_id"],
    "brand": laneige_cream_config["brand"],
    "product_name": laneige_cream_config["product_name"],
    "segment": laneige_cream_config["segment"],
    "sku": product["sku"],
    "current_price": 38.0,
    "currency": "USD",
    "size_ml": 50,
    "variant_name": "Matin Kim 50mL",
    "purchase_type": "One-time purchase",
    "price_source": "User-provided product-page screenshot, 2026-09-28",
    "description": product.get("description", "").strip(),
    "product_url": laneige_cream_config["product_url"],
    "variant_url": product["url"],
    "key_ingredients": None,
    "key_benefits": "Barrier support; Intense, long-lasting hydration",
    "skin_concern": None
}

laneige_cream_products_df = pd.DataFrame([
    laneige_cream_product_record
])

# 2. Extract the ingredient list from the selected product block.
properties = product.get("additionalProperty", [])
if isinstance(properties, dict):
    properties = [properties]

ingredient_values = [
    item["value"]
    for item in properties
    if item.get("name", "").casefold() == "ingredients"
    and isinstance(item.get("value"), str)
]

if len(ingredient_values) != 1:
    raise ValueError("Expected one full ingredient list.")

laneige_cream_ingredients_raw = ingredient_values[0]

# Remove the final punctuation while preserving the raw source above.
ingredient_names = [
    name.strip()
    for name in re.split(
        r",\s+",
        laneige_cream_ingredients_raw.strip().removesuffix(".")
    )
    if name.strip()
]

assert len(ingredient_names) == 38
assert "1,2-Hexanediol" in ingredient_names

laneige_cream_ingredients_df = pd.DataFrame({
    "product_id": laneige_cream_config["product_id"],
    "ingredient_position": range(1, len(ingredient_names) + 1),
    "ingredient_name": ingredient_names
})

# 3. Collect and clean reviews.
laneige_cream_reviews_url = (
    "https://api-cdn.yotpo.com/v3/storefront/store/"
    "pc9NUmcX8nZOBN32faV43yrNlBhSI07Q5DPJm1uE/"
    "product/7150484062260/reviews"
)

laneige_cream_reviews = collect_laneige_reviews(
    reviews_url=laneige_cream_reviews_url,
    product_id="LANEIGE_MOISTURIZER_01",
    max_reviews=300
)

if not laneige_cream_reviews:
    raise ValueError("No cream reviews returned.")

laneige_cream_reviews_df = pd.DataFrame([
    clean_laneige_review(review)
    for review in laneige_cream_reviews
])

laneige_cream_reviews_df["review_date"] = pd.to_datetime(
    laneige_cream_reviews_df["review_date"],
    errors="coerce"
)

assert laneige_cream_reviews_df["review_id"].notna().all()
assert laneige_cream_reviews_df["review_id"].is_unique
assert laneige_cream_reviews_df["review_date"].notna().all()
assert laneige_cream_reviews_df["review_date"].is_monotonic_decreasing

print("\nProduct: Matin Kim 50 ml — 38.00 USD")
print("Ingredients:", len(laneige_cream_ingredients_df))
print("Reviews:", len(laneige_cream_reviews_df))
print("All review checks passed.")

field_counts = Counter(
    field["title"].strip()
    for review in laneige_cream_reviews
    for field in (review.get("customFields") or {}).values()
    if field.get("title")
)

print("\nReviewer field labels:")
for title, count in field_counts.most_common():
    print(f"{title}: {count}")

In [ ]:
from pathlib import Path

# Create the concern table, preserving headers even when empty.
concerns = (
    laneige_cream_reviews_df[["review_id", "skin_concerns"]]
    .explode("skin_concerns")
    .dropna(subset=["skin_concerns"])
    .rename(columns={"skin_concerns": "skin_concern"})
    .copy()
)

concerns["skin_concern"] = (
    concerns["skin_concern"].astype(str).str.strip()
)

laneige_cream_review_skin_concerns_df = (
    concerns[concerns["skin_concern"] != ""]
    .drop_duplicates(["review_id", "skin_concern"])
    .reset_index(drop=True)
)

laneige_cream_reviews_clean_df = laneige_cream_reviews_df.drop(
    columns=["skin_concerns"]
).copy()

# Validate all four tables.
product_id = "LANEIGE_MOISTURIZER_01"
products = laneige_cream_products_df
reviews = laneige_cream_reviews_clean_df
ingredients = laneige_cream_ingredients_df
concerns = laneige_cream_review_skin_concerns_df

assert products["product_id"].tolist() == [product_id]
assert len(reviews) == 300
assert reviews["review_id"].notna().all()
assert reviews["review_id"].is_unique
assert reviews["product_id"].eq(product_id).all()
assert reviews["review_date"].notna().all()
assert reviews["review_date"].is_monotonic_decreasing
assert len(ingredients) == 38
assert ingredients["product_id"].eq(product_id).all()
assert concerns["review_id"].isin(reviews["review_id"]).all()
assert not concerns.duplicated(["review_id", "skin_concern"]).any()

laneige_cream_tables = {
    "products": products,
    "reviews": reviews,
    "ingredients": ingredients,
    "review_skin_concerns": concerns
}

output_folder = Path("data/processed") / product_id
output_folder.mkdir(parents=True, exist_ok=True)

for name, table in laneige_cream_tables.items():
    file_path = output_folder / f"{name}.csv"
    table.to_csv(file_path, index=False)
    assert len(pd.read_csv(file_path)) == len(table)
    print(f"{name}.csv: {len(table)} rows saved")

print("\nAll checks passed.")
print("Saved to:", output_folder.resolve())

In [ ]:
laneige_serum_config = next(
    product for product in products_config
    if product["product_id"] == "LANEIGE_SERUM_01"
)

laneige_serum_response = requests.get(
    laneige_serum_config["product_url"],
    timeout=30
)
laneige_serum_response.raise_for_status()

laneige_serum_soup = BeautifulSoup(
    laneige_serum_response.text,
    "html.parser"
)

laneige_serum_product_blocks = list_product_blocks(
    laneige_serum_soup
)

print("Final page URL:", laneige_serum_response.url)
print("Product blocks found:", len(laneige_serum_product_blocks))

for index, product in enumerate(laneige_serum_product_blocks):
    print(f"\n--- Product block {index} ---")
    print(json.dumps(product, indent=2, ensure_ascii=False))

In [ ]:
matches = [
    product for product in laneige_serum_product_blocks
    if product.get("sku") == "270284105"
]

if len(matches) != 1:
    raise ValueError("Expected one serum product block.")

laneige_serum_product_data = matches[0]

properties = laneige_serum_product_data.get("additionalProperty", [])
if isinstance(properties, dict):
    properties = [properties]

ingredient_values = [
    item["value"]
    for item in properties
    if item.get("name", "").casefold() == "ingredients"
    and isinstance(item.get("value"), str)
]

if len(ingredient_values) != 1:
    raise ValueError("Expected one ingredient list.")

# Preserve the complete source, including its disclaimer.
laneige_serum_ingredients_raw = ingredient_values[0]

# Remove the disclaimer and invisible formatting character.
formula_text = re.split(
    r"For informational purposes only",
    laneige_serum_ingredients_raw,
    maxsplit=1,
    flags=re.IGNORECASE
)[0]

formula_text = formula_text.replace("\ufeff", "").strip().removesuffix(".")

ingredient_names = [
    name.strip()
    for name in re.split(r",\s+", formula_text)
    if name.strip()
]

assert len(ingredient_names) == 30, "Unexpected ingredient count."
assert "1,2-Hexanediol" in ingredient_names

laneige_serum_ingredients_df = pd.DataFrame({
    "product_id": laneige_serum_config["product_id"],
    "ingredient_position": range(1, len(ingredient_names) + 1),
    "ingredient_name": ingredient_names
})

print("Ingredients:", len(laneige_serum_ingredients_df))
print("Last ingredient:", ingredient_names[-1])

laneige_serum_ingredients_df

In [ ]:
from collections import Counter

laneige_serum_product_record = {
    "product_id": laneige_serum_config["product_id"],
    "brand": laneige_serum_config["brand"],
    "product_name": laneige_serum_config["product_name"],
    "segment": laneige_serum_config["segment"],
    "sku": laneige_serum_product_data["sku"],
    "current_price": 45.0,
    "currency": "USD",
    "size_ml": 50,
    "purchase_type": "One-time purchase",
    "price_source": "User-provided product-page screenshot, 2026-09-28",
    "description": laneige_serum_product_data.get(
        "description", ""
    ).strip(),
    "product_url": laneige_serum_config["product_url"],
    "variant_url": (
        "https://us.laneige.com/products/"
        "water-bank-blue-hyaluronic-serum?variant=39875620864052"
    ),
    "key_ingredients": None,
    "key_benefits": (
        "Intense hydration; Smooth, supple complexion; "
        "Visibly firm and improve elasticity"
    ),
    "skin_concern": None
}

laneige_serum_products_df = pd.DataFrame([
    laneige_serum_product_record
])

# Start from page 1 using LANEIGE's newest-first sorting.
laneige_serum_reviews_url = (
    "https://api-cdn.yotpo.com/v3/storefront/store/"
    "pc9NUmcX8nZOBN32faV43yrNlBhSI07Q5DPJm1uE/"
    "product/6710563766324/reviews"
)

laneige_serum_reviews = collect_laneige_reviews(
    reviews_url=laneige_serum_reviews_url,
    product_id="LANEIGE_SERUM_01",
    max_reviews=300
)

if not laneige_serum_reviews:
    raise ValueError("No serum reviews returned.")

laneige_serum_reviews_df = pd.DataFrame([
    clean_laneige_review(review)
    for review in laneige_serum_reviews
])

laneige_serum_reviews_df["review_date"] = pd.to_datetime(
    laneige_serum_reviews_df["review_date"],
    errors="coerce"
)

assert laneige_serum_reviews_df["review_id"].notna().all()
assert laneige_serum_reviews_df["review_id"].is_unique
assert laneige_serum_reviews_df["review_date"].notna().all()
assert laneige_serum_reviews_df["review_date"].is_monotonic_decreasing
assert laneige_serum_reviews_df["product_id"].eq("LANEIGE_SERUM_01").all()

print("\nProduct: Water Bank Blue Hyaluronic Serum")
print("Size / price: 50 ml / 45.00 USD")
print("Ingredients:", len(laneige_serum_ingredients_df))
print("Reviews:", len(laneige_serum_reviews_df))
print("All review checks passed.")

field_counts = Counter(
    field["title"].strip()
    for review in laneige_serum_reviews
    for field in (review.get("customFields") or {}).values()
    if field.get("title")
)

print("\nReviewer field labels:")
for title, count in field_counts.most_common():
    print(f"{title}: {count}")

In [ ]:
from pathlib import Path

# Separate reported skin concerns.
concerns = (
    laneige_serum_reviews_df[["review_id", "skin_concerns"]]
    .explode("skin_concerns")
    .dropna(subset=["skin_concerns"])
    .rename(columns={"skin_concerns": "skin_concern"})
    .copy()
)

concerns["skin_concern"] = (
    concerns["skin_concern"].astype(str).str.strip()
)

laneige_serum_review_skin_concerns_df = (
    concerns[concerns["skin_concern"] != ""]
    .drop_duplicates(["review_id", "skin_concern"])
    .reset_index(drop=True)
)

laneige_serum_reviews_clean_df = laneige_serum_reviews_df.drop(
    columns=["skin_concerns"]
).copy()

# Validate all four tables.
product_id = "LANEIGE_SERUM_01"
products = laneige_serum_products_df
reviews = laneige_serum_reviews_clean_df
ingredients = laneige_serum_ingredients_df
concerns = laneige_serum_review_skin_concerns_df

assert products["product_id"].tolist() == [product_id]
assert len(reviews) == 300
assert reviews["review_id"].notna().all()
assert reviews["review_id"].is_unique
assert reviews["product_id"].eq(product_id).all()
assert reviews["review_date"].notna().all()
assert reviews["review_date"].is_monotonic_decreasing
assert len(ingredients) == 30
assert ingredients["product_id"].eq(product_id).all()
assert concerns["review_id"].isin(reviews["review_id"]).all()
assert not concerns.duplicated(["review_id", "skin_concern"]).any()

laneige_serum_tables = {
    "products": products,
    "reviews": reviews,
    "ingredients": ingredients,
    "review_skin_concerns": concerns
}

output_folder = Path("data/processed") / product_id
output_folder.mkdir(parents=True, exist_ok=True)

for name, table in laneige_serum_tables.items():
    file_path = output_folder / f"{name}.csv"
    table.to_csv(file_path, index=False)
    assert len(pd.read_csv(file_path)) == len(table)
    print(f"{name}.csv: {len(table)} rows saved")

print(
    "\nReviews with skin concerns:",
    concerns["review_id"].nunique()
)
print("All checks passed.")
print("Saved to:", output_folder.resolve())

concerns

In [ ]:
def split_skin_concerns(value):
    if value is None:
        return []

    values = value if isinstance(value, list) else [value]

    return [
        concern.strip()
        for item in values
        if isinstance(item, str)
        for concern in item.split(",")
        if concern.strip()
    ]


concerns = laneige_serum_reviews_df[
    ["review_id", "skin_concerns"]
].copy()

concerns["skin_concerns"] = concerns["skin_concerns"].apply(
    split_skin_concerns
)

laneige_serum_review_skin_concerns_df = (
    concerns
    .explode("skin_concerns")
    .dropna(subset=["skin_concerns"])
    .rename(columns={"skin_concerns": "skin_concern"})
    .drop_duplicates(["review_id", "skin_concern"])
    .reset_index(drop=True)
)

concerns = laneige_serum_review_skin_concerns_df

assert concerns["review_id"].isin(
    laneige_serum_reviews_clean_df["review_id"]
).all()
assert not concerns.duplicated(["review_id", "skin_concern"]).any()
assert not concerns["skin_concern"].str.contains(",", regex=False).any()

laneige_serum_tables["review_skin_concerns"] = concerns

file_path = (
    Path("data/processed/LANEIGE_SERUM_01")
    / "review_skin_concerns.csv"
)
concerns.to_csv(file_path, index=False)

assert len(pd.read_csv(file_path)) == len(concerns)

print("Review–concern pairs:", len(concerns))
print("Reviews with skin concerns:", concerns["review_id"].nunique())
print("\n" + concerns.to_string(index=False))

In [ ]:
laneige_toner_config = next(
    product for product in products_config
    if product["product_id"] == "LANEIGE_TONER_01"
)

laneige_toner_response = requests.get(
    laneige_toner_config["product_url"],
    timeout=30
)
laneige_toner_response.raise_for_status()

laneige_toner_soup = BeautifulSoup(
    laneige_toner_response.text,
    "html.parser"
)

laneige_toner_product_blocks = list_product_blocks(
    laneige_toner_soup
)

print("Final page URL:", laneige_toner_response.url)
print("Product blocks found:", len(laneige_toner_product_blocks))

for index, product in enumerate(laneige_toner_product_blocks):
    print(f"\n--- Product block {index} ---")
    print(json.dumps(product, indent=2, ensure_ascii=False))

In [ ]:
from urllib.parse import urlparse, parse_qs
from collections import Counter
from html import unescape

# 1. Select the variant confirmed by your screenshot.
matches = [
    product for product in laneige_toner_product_blocks
    if parse_qs(
        urlparse(product.get("url", "")).query
    ).get("variant", [None])[0] == "48218688585780"
]

if len(matches) != 1:
    raise ValueError("Expected one Matin Kim 80 ml product block.")

product = matches[0]

laneige_toner_product_record = {
    "product_id": laneige_toner_config["product_id"],
    "brand": laneige_toner_config["brand"],
    "product_name": laneige_toner_config["product_name"],
    "segment": laneige_toner_config["segment"],
    "sku": product["sku"],
    "current_price": 19.0,
    "currency": "USD",
    "size_ml": 80,
    "variant_name": "Matin Kim 80mL",
    "purchase_type": "One-time purchase",
    "price_source": "User-provided product-page screenshot, 2026-09-28",
    "description": unescape(product.get("description", "")).strip(),
    "product_url": laneige_toner_config["product_url"],
    "variant_url": product["url"],
    "key_ingredients": "Ceramide & peptide complex; White Leaf Tea",
    "key_benefits": (
        "Hydration; Barrier support; Nourish and visibly firm; Soothe"
    ),
    "skin_concern": None
}

laneige_toner_products_df = pd.DataFrame([
    laneige_toner_product_record
])

# 2. Extract the selected variant's ingredients.
properties = product.get("additionalProperty", [])
if isinstance(properties, dict):
    properties = [properties]

ingredient_values = [
    item["value"]
    for item in properties
    if item.get("name", "").casefold() == "ingredients"
    and isinstance(item.get("value"), str)
]

if len(ingredient_values) != 1:
    raise ValueError("Expected one full ingredient list.")

laneige_toner_ingredients_raw = ingredient_values[0]

ingredient_names = [
    name.strip()
    for name in re.split(
        r",\s+",
        laneige_toner_ingredients_raw.strip().removesuffix(".")
    )
    if name.strip()
]

assert len(ingredient_names) == 26
assert "1,2-Hexanediol" in ingredient_names

laneige_toner_ingredients_df = pd.DataFrame({
    "product_id": laneige_toner_config["product_id"],
    "ingredient_position": range(1, len(ingredient_names) + 1),
    "ingredient_name": ingredient_names
})

# 3. Collect reviews from page 1, newest first.
laneige_toner_reviews_url = (
    "https://api-cdn.yotpo.com/v3/storefront/store/"
    "pc9NUmcX8nZOBN32faV43yrNlBhSI07Q5DPJm1uE/"
    "product/7091910148148/reviews"
)

laneige_toner_reviews = collect_laneige_reviews(
    reviews_url=laneige_toner_reviews_url,
    product_id="LANEIGE_TONER_01",
    max_reviews=300
)

if not laneige_toner_reviews:
    raise ValueError("No toner reviews returned.")

laneige_toner_reviews_df = pd.DataFrame([
    clean_laneige_review(review)
    for review in laneige_toner_reviews
])

laneige_toner_reviews_df["review_date"] = pd.to_datetime(
    laneige_toner_reviews_df["review_date"],
    errors="coerce"
)

# 4. Split individual skin concerns; keep skincare goals separate.
concerns = laneige_toner_reviews_df[
    ["review_id", "skin_concerns"]
].copy()

concerns["skin_concerns"] = concerns["skin_concerns"].apply(
    split_skin_concerns
)

laneige_toner_review_skin_concerns_df = (
    concerns
    .explode("skin_concerns")
    .dropna(subset=["skin_concerns"])
    .rename(columns={"skin_concerns": "skin_concern"})
    .drop_duplicates(["review_id", "skin_concern"])
    .reset_index(drop=True)
)

laneige_toner_reviews_clean_df = laneige_toner_reviews_df.drop(
    columns=["skin_concerns"]
).copy()

# 5. Validate and inspect before saving.
reviews = laneige_toner_reviews_clean_df
concerns = laneige_toner_review_skin_concerns_df

assert reviews["review_id"].notna().all()
assert reviews["review_id"].is_unique
assert reviews["product_id"].eq("LANEIGE_TONER_01").all()
assert reviews["review_date"].notna().all()
assert reviews["review_date"].is_monotonic_decreasing
assert concerns["review_id"].isin(reviews["review_id"]).all()

print("\nProduct: Cream Skin — Matin Kim 80 ml / 19.00 USD")
print("Ingredients:", len(laneige_toner_ingredients_df))
print("Reviews:", len(reviews))
print("Review–concern pairs:", len(concerns))
print("Reviews with skin concerns:", concerns["review_id"].nunique())
print("All review checks passed.")

field_counts = Counter(
    field["title"].strip()
    for review in laneige_toner_reviews
    for field in (review.get("customFields") or {}).values()
    if field.get("title")
)

print("\nReviewer field labels:")
for title, count in field_counts.most_common():
    print(f"{title}: {count}")

In [ ]:
from pathlib import Path

# 1. Validate and save the toner.
product_id = "LANEIGE_TONER_01"

laneige_toner_tables = {
    "products": laneige_toner_products_df,
    "reviews": laneige_toner_reviews_clean_df,
    "ingredients": laneige_toner_ingredients_df,
    "review_skin_concerns": laneige_toner_review_skin_concerns_df
}

assert laneige_toner_products_df["product_id"].tolist() == [product_id]
assert len(laneige_toner_reviews_clean_df) == 300
assert laneige_toner_reviews_clean_df["product_id"].eq(product_id).all()
assert len(laneige_toner_ingredients_df) == 26
assert laneige_toner_ingredients_df["product_id"].eq(product_id).all()

toner_folder = Path("data/processed") / product_id
toner_folder.mkdir(parents=True, exist_ok=True)

for name, table in laneige_toner_tables.items():
    file_path = toner_folder / f"{name}.csv"
    table.to_csv(file_path, index=False)
    assert len(pd.read_csv(file_path)) == len(table)

print("Toner's four tables saved.")

# 2. Read only the four individual LANEIGE product folders.
laneige_product_ids = [
    "LANEIGE_CLEANSER_01",
    "LANEIGE_MOISTURIZER_01",
    "LANEIGE_SERUM_01",
    "LANEIGE_TONER_01"
]

table_names = [
    "products",
    "reviews",
    "ingredients",
    "review_skin_concerns"
]

required_files = [
    Path("data/processed") / pid / f"{name}.csv"
    for pid in laneige_product_ids
    for name in table_names
]

missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise ValueError("Missing files:\n" + "\n".join(missing))

laneige_tables = {}

for name in table_names:
    laneige_tables[name] = pd.concat(
        [
            pd.read_csv(
                Path("data/processed") / pid / f"{name}.csv",
                dtype={"product_id": str, "review_id": str, "sku": str}
            )
            for pid in laneige_product_ids
        ],
        ignore_index=True
    )

# 3. Validate the combined tables.
products = laneige_tables["products"]
reviews = laneige_tables["reviews"]
ingredients = laneige_tables["ingredients"]
concerns = laneige_tables["review_skin_concerns"]

assert products["product_id"].is_unique
assert set(products["product_id"]) == set(laneige_product_ids)
assert reviews["review_id"].notna().all()
assert reviews["review_id"].is_unique, (
    "Review IDs overlap across products. Inspect before combining."
)
assert reviews["product_id"].isin(products["product_id"]).all()
assert ingredients["product_id"].isin(products["product_id"]).all()
assert concerns["review_id"].isin(reviews["review_id"]).all()
assert not ingredients.duplicated(
    ["product_id", "ingredient_position"]
).any()
assert not concerns.duplicated(["review_id", "skin_concern"]).any()

for pid, group in reviews.groupby("product_id"):
    dates = pd.to_datetime(group["review_date"], errors="coerce")
    assert dates.notna().all(), f"Invalid dates: {pid}"
    assert dates.is_monotonic_decreasing, f"Date order issue: {pid}"

# 4. Save the combined brand tables.
output_folder = Path("data/processed/laneige")
output_folder.mkdir(parents=True, exist_ok=True)

for name, table in laneige_tables.items():
    file_path = output_folder / f"{name}.csv"
    table.to_csv(file_path, index=False)
    assert len(pd.read_csv(file_path)) == len(table)
    print(f"{name}.csv: {len(table)} rows saved")

print("\nReviews per product:")
print(reviews.groupby("product_id").size().to_string())

print("\nAll combined checks passed.")
print("Saved to:", output_folder.resolve())

In [ ]:
from pathlib import Path

brand_folders = ["beauty_of_joseon", "cosrx", "laneige"]
table_names = [
    "products", "reviews", "ingredients", "review_skin_concerns"
]

# Read only the combined brand folders to avoid counting products twice.
master_tables = {}

for name in table_names:
    frames = []
    for brand_folder in brand_folders:
        file_path = Path("data/processed") / brand_folder / f"{name}.csv"

        if not file_path.exists():
            raise FileNotFoundError(file_path)

        frames.append(pd.read_csv(
            file_path,
            dtype={"product_id": str, "review_id": str, "sku": str}
        ))

    master_tables[name] = pd.concat(frames, ignore_index=True)

products = master_tables["products"]
reviews = master_tables["reviews"]
ingredients = master_tables["ingredients"]
concerns = master_tables["review_skin_concerns"]

# Preserve original segment labels and add a consistent analysis category.
products["analysis_segment"] = products["segment"].replace({
    "Toner / Essence": "Toner"
})

# Validate IDs and relationships.
expected_ids = {product["product_id"] for product in products_config}

assert len(products) == 12
assert products["product_id"].is_unique
assert set(products["product_id"]) == expected_ids

assert reviews["review_id"].notna().all()
assert reviews["review_id"].is_unique, (
    "Review IDs overlap across products or brands. Inspect before saving."
)
assert reviews["product_id"].isin(expected_ids).all()
assert ingredients["product_id"].isin(expected_ids).all()
assert concerns["review_id"].isin(reviews["review_id"]).all()

assert not ingredients.duplicated(
    ["product_id", "ingredient_position"]
).any()
assert not concerns.duplicated(["review_id", "skin_concern"]).any()
assert concerns["skin_concern"].notna().all()

# Check review counts, ratings, and ordering within each product.
review_counts = reviews.groupby("product_id").size()
assert set(review_counts.index) == expected_ids
assert review_counts.eq(300).all()

reviews["rating"] = pd.to_numeric(reviews["rating"], errors="coerce")
assert reviews["rating"].between(1, 5).all()

for product_id, group in reviews.groupby("product_id"):
    dates = pd.to_datetime(group["review_date"], errors="coerce")
    assert dates.notna().all(), f"Invalid dates: {product_id}"
    assert dates.is_monotonic_decreasing, f"Date order issue: {product_id}"

# Validate ingredient positions for every product.
assert set(ingredients["product_id"]) == expected_ids

for product_id, group in ingredients.groupby("product_id"):
    positions = pd.to_numeric(
        group["ingredient_position"], errors="raise"
    ).sort_values().tolist()
    assert positions == list(range(1, len(group) + 1)), (
        f"Ingredient position issue: {product_id}"
    )

# Save the master tables separately.
master_folder = Path("data/processed/all_products")
master_folder.mkdir(parents=True, exist_ok=True)

for name, table in master_tables.items():
    file_path = master_folder / f"{name}.csv"
    table.to_csv(file_path, index=False)
    assert len(pd.read_csv(file_path)) == len(table)
    print(f"{name}.csv: {len(table)} rows saved")

print("\nProducts by brand and segment:")
print(pd.crosstab(
    products["brand"], products["analysis_segment"]
).to_string())

print("\nAll master checks passed.")
print("Saved to:", master_folder.resolve())

In [ ]:
from pathlib import Path
import pandas as pd

brand_folders = ["beauty_of_joseon", "cosrx", "laneige"]
table_names = [
    "products",
    "reviews",
    "ingredients",
    "review_skin_concerns"
]

# 1. Read only the combined brand folders.
master_tables = {}

for name in table_names:
    frames = []

    for brand_folder in brand_folders:
        file_path = (
            Path("data/processed") / brand_folder / f"{name}.csv"
        )

        if not file_path.exists():
            raise FileNotFoundError(file_path)

        frames.append(pd.read_csv(
            file_path,
            dtype={
                "product_id": str,
                "review_id": str,
                "sku": str
            }
        ))

    master_tables[name] = pd.concat(frames, ignore_index=True)

products = master_tables["products"]
reviews = master_tables["reviews"]
ingredients = master_tables["ingredients"]
concerns = master_tables["review_skin_concerns"]

# 2. Preserve source segments and add a consistent analysis category.
products["analysis_segment"] = products["segment"].replace({
    "Toner / Essence": "Toner"
})

expected_ids = {product["product_id"] for product in products_config}

assert len(expected_ids) == 12, "Expected 12 configured product IDs."
assert len(products) == 12, "Expected 12 product rows."
assert products["product_id"].is_unique, "Duplicate product IDs."
assert set(products["product_id"]) == expected_ids

# 3. Validate review IDs, relationships, counts, and ratings.
assert reviews["review_id"].notna().all(), "Missing review IDs."
assert reviews["review_id"].is_unique, (
    "Review IDs overlap across products or brands. Inspect before saving."
)
assert reviews["product_id"].isin(expected_ids).all(), (
    "Reviews reference unknown products."
)

review_counts = reviews.groupby("product_id").size()
assert set(review_counts.index) == expected_ids
assert review_counts.eq(300).all(), (
    "Expected 300 reviews for each product.\n"
    + review_counts.to_string()
)

reviews["rating"] = pd.to_numeric(
    reviews["rating"], errors="coerce"
)
assert reviews["rating"].between(1, 5).all(), (
    "Missing or invalid review ratings."
)

# 4. Parse dates individually to support mixed timestamp formatting.
# Do not assign a timezone or overwrite the original date column.
for product_id, group in reviews.groupby("product_id"):
    dates = [
        pd.to_datetime(value, errors="coerce")
        for value in group["review_date"]
    ]

    invalid = [pd.isna(value) for value in dates]

    if any(invalid):
        print(
            group.loc[invalid, ["review_id", "review_date"]]
            .to_string(index=False)
        )
        raise ValueError(f"Unparseable dates: {product_id}")

    timezone_present = [
        value.tzinfo is not None for value in dates
    ]

    if len(set(timezone_present)) > 1:
        raise ValueError(
            f"Mixed timezone-aware and timezone-free dates: {product_id}"
        )

    assert all(
        earlier >= later
        for earlier, later in zip(dates, dates[1:])
    ), f"Reviews are not newest first: {product_id}"

# 5. Validate ingredients and their positions.
assert ingredients["product_id"].isin(expected_ids).all()
assert set(ingredients["product_id"]) == expected_ids
assert ingredients["ingredient_name"].notna().all()
assert ingredients["ingredient_name"].str.strip().ne("").all()

ingredients["ingredient_position"] = pd.to_numeric(
    ingredients["ingredient_position"], errors="raise"
)

assert not ingredients.duplicated(
    ["product_id", "ingredient_position"]
).any(), "Duplicate ingredient positions."

for product_id, group in ingredients.groupby("product_id"):
    positions = group["ingredient_position"].sort_values().tolist()

    assert positions == list(range(1, len(group) + 1)), (
        f"Missing or invalid ingredient positions: {product_id}"
    )

# 6. Validate review–concern relationships.
assert concerns["review_id"].isin(reviews["review_id"]).all(), (
    "Concerns reference unknown reviews."
)
assert concerns["skin_concern"].notna().all()
assert concerns["skin_concern"].str.strip().ne("").all()
assert not concerns.duplicated(
    ["review_id", "skin_concern"]
).any(), "Duplicate review–concern pairs."

# Confirm one product for each brand/segment combination.
brand_segment_counts = pd.crosstab(
    products["brand"],
    products["analysis_segment"]
)

assert brand_segment_counts.shape == (3, 4)
assert brand_segment_counts.eq(1).all().all(), (
    "Expected one product per brand and segment."
)

# 7. Save only after all validations pass.
master_folder = Path("data/processed/all_products")
master_folder.mkdir(parents=True, exist_ok=True)

for name, table in master_tables.items():
    file_path = master_folder / f"{name}.csv"
    table.to_csv(file_path, index=False)

    saved_table = pd.read_csv(file_path)
    assert len(saved_table) == len(table), (
        f"Saved row count mismatch: {name}"
    )

    print(f"{name}.csv: {len(table)} rows saved")

print("\nProducts by brand and segment:")
print(brand_segment_counts.to_string())

print("\nReviews per product:")
print(review_counts.to_string())

print("\nAll master checks passed.")
print("Saved to:", master_folder.resolve())